<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/11_s_slte_downstream_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
from pathlib import Path
import hashlib
import json
import pandas as pd

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load frozen research manifests.
with (BASE / "configs/holdout_partition_manifest_v1.json").open() as f:
    partition_manifest = json.load(f)

with (BASE / "outputs/llm_main_v1/full_annotation_audit_v1.json").open() as f:
    annotation_audit = json.load(f)

with (BASE / "outputs/s_slte_evidence_v1/evidence_manifest_v1.json").open() as f:
    evidence_manifest = json.load(f)

with (BASE / "outputs/s_slte_calibration_v1/frozen_correction_rule_v1.json").open() as f:
    frozen_rule = json.load(f)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def id_fingerprint(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

# Validate the original annotation audit.
assert annotation_audit["status"] == "FULL_CHECKPOINT_AUDIT_PASSED"
assert annotation_audit["total_records"] == 15621
assert annotation_audit["invalid_predictions"] == 0

# Restore the exact frozen active-training IDs.
training_spec = partition_manifest["partitions"]["training"]

training_ids = set(map(int, training_spec["record_ids"]))

assert len(training_ids) == training_spec["count"] == 13621
assert id_fingerprint(training_ids) == training_spec["id_sha256"]

# Load only record IDs and original text, never reference labels.
source_training = pd.read_csv(
    BASE / "data/splits/noisy_training.csv",
    usecols=["record_id", "text"]
)

assert source_training["record_id"].is_unique
assert source_training["text"].notna().all()
assert training_ids.issubset(
    set(source_training["record_id"].astype(int))
)

# Filter to the frozen 13,621 active training records.
training_text = (
    source_training.loc[
        source_training["record_id"].isin(training_ids)
    ]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(training_text) == 13621
assert set(training_text["record_id"].astype(int)) == training_ids
assert text_fingerprint(training_text) == (
    training_spec["raw_text_sha256"]
)

# Load and verify original LLM annotations.
annotation_path = (
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)

assert sha256_file(annotation_path) == (
    annotation_audit["exports"]["training"]["sha256"]
)

annotations = pd.read_csv(annotation_path)

assert len(annotations) == 13621
assert annotations["record_id"].is_unique
assert set(annotations["record_id"].astype(int)) == training_ids
assert annotations["llm_label"].isin(range(4)).all()

# Load and verify saved semantic neighborhood evidence.
evidence_path = (
    BASE / "outputs/s_slte_evidence_v1/training_neighbor_evidence_v1.csv"
)

assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["training"]["sha256"]
)

evidence = pd.read_csv(evidence_path)

assert len(evidence) == 13621
assert evidence["record_id"].is_unique
assert set(evidence["record_id"].astype(int)) == training_ids
assert evidence["has_positive_similarity"].all()
assert evidence["neighbor_top_label"].isin(range(4)).all()

for class_id in range(4):
    assert f"support_class_{class_id}" in evidence.columns

# Verify alignment between LLM labels and semantic evidence.
aligned = annotations[["record_id", "llm_label"]].merge(
    evidence[["record_id", "llm_label"]],
    on="record_id",
    validate="one_to_one",
    suffixes=("_annotation", "_evidence")
)

assert (
    aligned["llm_label_annotation"]
    == aligned["llm_label_evidence"]
).all()

# Verify the previously frozen correction rule.
assert frozen_rule["method"] == (
    "S_SLTE_SEMANTIC_ONLY_CONSERVATIVE_RULE"
)
assert frozen_rule["min_neighbor_support"] == 0.80
assert frozen_rule["min_semantic_support_gap"] == 0.50
assert frozen_rule["k"] == 20
assert frozen_rule["training_labels_modified"] is False

# Create a text-and-LLM-label dataset for the next step.
training_df = training_text.merge(
    annotations[["record_id", "llm_label"]],
    on="record_id",
    validate="one_to_one"
)

assert len(training_df) == 13621
assert "label" not in training_df.columns

print("STAGE 13 — INPUT RESTORATION")

print("\nSOURCE DATA")
print("Original source rows:", len(source_training))
print("Frozen active training rows:", len(training_text))
print("Excluded source rows:", len(source_training) - len(training_text))

print("\nVERIFICATION")
print("Frozen ID fingerprint: VERIFIED")
print("Frozen text fingerprint: VERIFIED")
print("LLM annotations: VERIFIED")
print("Semantic evidence: VERIFIED")
print("Record ID alignment: VERIFIED")
print("Original annotation audit: VERIFIED")

print("\nFROZEN S-SLTE RULE")
print("Minimum support:", frozen_rule["min_neighbor_support"])
print("Minimum gap:", frozen_rule["min_semantic_support_gap"])
print("Clean anchor K:", frozen_rule["k"])

print("\nRESEARCH INTEGRITY")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Training labels modified: NO")
print("New inference performed: NO")

print("\nSTATUS: STAGE 13 INPUT VERIFICATION PASSED")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 13 — INPUT RESTORATION

SOURCE DATA
Original source rows: 16200
Frozen active training rows: 13621
Excluded source rows: 2579

VERIFICATION
Frozen ID fingerprint: VERIFIED
Frozen text fingerprint: VERIFIED
LLM annotations: VERIFIED
Semantic evidence: VERIFIED
Record ID alignment: VERIFIED
Original annotation audit: VERIFIED

FROZEN S-SLTE RULE
Minimum support: 0.8
Minimum gap: 0.5
Clean anchor K: 20

RESEARCH INTEGRITY
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Training labels modified: NO
New inference performed: NO

STATUS: STAGE 13 INPUT VERIFICATION PASSED


In [ ]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Stage 13 — Frozen S-SLTE training correction.
OUTPUT_DIR = BASE / "outputs/s_slte_training_correction_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RULE_PATH = (
    BASE / "outputs/s_slte_calibration_v1"
    / "frozen_correction_rule_v1.json"
)

ANNOTATION_PATH = (
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)

EVIDENCE_PATH = (
    BASE / "outputs/s_slte_evidence_v1"
    / "training_neighbor_evidence_v1.csv"
)

assert len(training_df) == 13621
assert len(evidence) == 13621
assert training_df["record_id"].is_unique
assert evidence["record_id"].is_unique
assert "label" not in training_df.columns
assert "label" not in evidence.columns

assert frozen_rule["min_neighbor_support"] == 0.80
assert frozen_rule["min_semantic_support_gap"] == 0.50
assert frozen_rule["k"] == 20

assert sha256_file(ANNOTATION_PATH) == (
    annotation_audit["exports"]["training"]["sha256"]
)

assert sha256_file(EVIDENCE_PATH) == (
    evidence_manifest["partitions"]["training"]["sha256"]
)

with RULE_PATH.open(encoding="utf-8") as f:
    saved_rule = json.load(f)

assert saved_rule == frozen_rule

SUPPORT_COLUMNS = [
    f"support_class_{class_id}"
    for class_id in range(4)
]

# Join existing LLM labels and anchor-neighborhood evidence.
combined = training_df[
    ["record_id", "llm_label"]
].merge(
    evidence[
        [
            "record_id",
            "llm_label",
            "neighbor_top_label",
            "neighbor_top_support",
            "llm_label_support",
            "has_positive_similarity",
            *SUPPORT_COLUMNS
        ]
    ],
    on=["record_id", "llm_label"],
    how="inner",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(combined) == 13621
assert set(combined["record_id"].astype(int)) == training_ids
assert combined["has_positive_similarity"].all()

llm = combined["llm_label"].to_numpy(dtype=int)

neighbor = combined[
    "neighbor_top_label"
].to_numpy(dtype=int)

support = combined[
    SUPPORT_COLUMNS
].to_numpy(dtype=np.float64)

assert np.isin(llm, range(4)).all()
assert np.isin(neighbor, range(4)).all()
assert np.isfinite(support).all()
assert np.all(support >= 0)
assert np.allclose(
    support.sum(axis=1), 1.0, atol=1e-6
)
assert np.array_equal(
    support.argmax(axis=1), neighbor
)

top_support = support.max(axis=1)
llm_support = support[np.arange(len(llm)), llm]

assert np.allclose(
    top_support,
    combined["neighbor_top_support"].to_numpy(),
    atol=1e-6
)

assert np.allclose(
    llm_support,
    combined["llm_label_support"].to_numpy(),
    atol=1e-6
)

support_gap = np.maximum(
    top_support - llm_support, 0.0
)

conflict = llm != neighbor

assert int(conflict.sum()) == 2590

# Apply the previously frozen rule without retuning.
should_correct = (
    conflict
    & (
        top_support >=
        frozen_rule["min_neighbor_support"]
    )
    & (
        support_gap >=
        frozen_rule["min_semantic_support_gap"]
    )
)

corrected = np.where(
    should_correct,
    neighbor,
    llm
)

assert np.array_equal(
    corrected[~should_correct],
    llm[~should_correct]
)

assert np.array_equal(
    corrected[should_correct],
    neighbor[should_correct]
)

assert np.isin(corrected, range(4)).all()

# Preserve decisions separately from the original annotations.
decisions = pd.DataFrame({
    "record_id": combined["record_id"].astype("int64"),
    "original_llm_label": llm,
    "neighbor_proposed_label": neighbor,
    "neighbor_top_support": top_support,
    "llm_label_support": llm_support,
    "semantic_support_gap": support_gap,
    "llm_neighbor_conflict": conflict,
    "correction_applied": should_correct,
    "corrected_label": corrected
})

corrected_labels = decisions[
    [
        "record_id",
        "original_llm_label",
        "corrected_label",
        "correction_applied"
    ]
].copy()

assert len(decisions) == len(corrected_labels) == 13621
assert decisions["record_id"].is_unique
assert corrected_labels["record_id"].is_unique

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

    assert path.read_bytes() == payload
    return "NEW_SAVED"

decisions_path = (
    OUTPUT_DIR / "training_correction_decisions_v1.csv"
)

labels_path = (
    OUTPUT_DIR / "training_corrected_labels_v1.csv"
)

decisions_status = save_immutable(
    decisions_path,
    decisions.to_csv(
        index=False,
        float_format="%.8f",
        lineterminator="\n"
    )
)

labels_status = save_immutable(
    labels_path,
    corrected_labels.to_csv(
        index=False,
        lineterminator="\n"
    )
)

correction_count = int(should_correct.sum())

class_names = [
    "World", "Sports", "Business", "Sci/Tech"
]

class_summary = []

for class_id, class_name in enumerate(class_names):
    class_summary.append({
        "class_name": class_name,
        "original_count": int((llm == class_id).sum()),
        "corrected_count": int((corrected == class_id).sum()),
        "incoming_corrections": int(
            (should_correct & (neighbor == class_id)).sum()
        ),
        "outgoing_corrections": int(
            (should_correct & (llm == class_id)).sum()
        )
    })

manifest = {
    "protocol_version": "v1",
    "stage": "FROZEN_TRAINING_LABEL_CORRECTION",
    "method": frozen_rule["method"],
    "total_records": 13621,
    "llm_neighbor_conflicts": int(conflict.sum()),
    "corrections_applied": correction_count,
    "original_labels_retained": int((~should_correct).sum()),
    "abstained_conflicts": int(
        (conflict & ~should_correct).sum()
    ),
    "min_support": 0.80,
    "min_gap": 0.50,
    "k": 20,
    "class_summary": class_summary,
    "source_sha256": {
        "frozen_rule": sha256_file(RULE_PATH),
        "llm_annotations": sha256_file(ANNOTATION_PATH),
        "semantic_evidence": sha256_file(EVIDENCE_PATH)
    },
    "output_sha256": {
        "decisions": sha256_file(decisions_path),
        "corrected_labels": sha256_file(labels_path)
    },
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "original_llm_annotations_modified": False,
    "downstream_classifier_trained": False,
    "correction_accuracy_measured": False
}

manifest_path = (
    OUTPUT_DIR / "training_correction_manifest_v1.json"
)

manifest_status = save_immutable(
    manifest_path,
    json.dumps(manifest, indent=2) + "\n"
)

print("STAGE 13 — TRAINING CORRECTION COMPLETED")

print("\nCORRECTION SUMMARY")
print("Training records:", len(corrected_labels))
print("LLM–Neighbor conflicts:", int(conflict.sum()))
print("Corrections applied:", correction_count)
print("Original labels retained:", int((~should_correct).sum()))
print(
    "Conflicts abstained:",
    int((conflict & ~should_correct).sum())
)
print(
    "Correction coverage:",
    f"{100 * correction_count / len(corrected_labels):.2f}%"
)

print("\nCLASS DISTRIBUTION")
display(pd.DataFrame(class_summary))

print("\nSAVED ARTIFACTS")
print("Decisions:", decisions_status)
print("Corrected labels:", labels_status)
print("Manifest:", manifest_status)
print("Directory:", OUTPUT_DIR)

print("\nRESEARCH INTEGRITY")
print("Frozen rule unchanged: YES")
print("Source fingerprints verified: YES")
print("Original annotation file modified: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Downstream model trained: NO")
print("Correction accuracy measured: NO")

print("\nSTATUS: STAGE 13 TRAINING CORRECTION PASSED")


STAGE 13 — TRAINING CORRECTION COMPLETED

CORRECTION SUMMARY
Training records: 13621
LLM–Neighbor conflicts: 2590
Corrections applied: 705
Original labels retained: 12916
Conflicts abstained: 1885
Correction coverage: 5.18%

CLASS DISTRIBUTION


,class_name,original_count,corrected_count,incoming_corrections,outgoing_corrections
0,World,4199,4008,10,201
1,Sports,3532,3542,49,39
2,Business,4015,3662,96,449
3,Sci/Tech,1875,2409,550,16



SAVED ARTIFACTS
Decisions: NEW_SAVED
Corrected labels: NEW_SAVED
Manifest: NEW_SAVED
Directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_training_correction_v1

RESEARCH INTEGRITY
Frozen rule unchanged: YES
Source fingerprints verified: YES
Original annotation file modified: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Downstream model trained: NO
Correction accuracy measured: NO

STATUS: STAGE 13 TRAINING CORRECTION PASSED


In [ ]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Stage 13 — Prepare matched downstream training datasets.
DOWNSTREAM_DIR = BASE / "outputs/s_slte_downstream_v1"
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

CORRECTION_DIR = BASE / "outputs/s_slte_training_correction_v1"

LABELS_PATH = (
    CORRECTION_DIR / "training_corrected_labels_v1.csv"
)

DECISIONS_PATH = (
    CORRECTION_DIR / "training_correction_decisions_v1.csv"
)

CORRECTION_MANIFEST_PATH = (
    CORRECTION_DIR / "training_correction_manifest_v1.json"
)

RULE_PATH = (
    BASE / "outputs/s_slte_calibration_v1"
    / "frozen_correction_rule_v1.json"
)

def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_bytes(payload)
    os.replace(temp, path)

    assert path.read_bytes() == payload
    return "NEW_SAVED"

# Verify saved correction artifacts.
with CORRECTION_MANIFEST_PATH.open(encoding="utf-8") as f:
    correction_manifest = json.load(f)

assert correction_manifest["total_records"] == 13621
assert correction_manifest["corrections_applied"] == 705

assert file_digest(LABELS_PATH) == (
    correction_manifest["output_sha256"]["corrected_labels"]
)

assert file_digest(DECISIONS_PATH) == (
    correction_manifest["output_sha256"]["decisions"]
)

assert file_digest(RULE_PATH) == (
    correction_manifest["source_sha256"]["frozen_rule"]
)

# Load the two saved correction outputs.
corrected_labels = pd.read_csv(LABELS_PATH)
saved_decisions = pd.read_csv(DECISIONS_PATH)

assert len(corrected_labels) == 13621
assert len(saved_decisions) == 13621
assert corrected_labels["record_id"].is_unique
assert saved_decisions["record_id"].is_unique

# Verify that both saved artifacts describe identical decisions.
decision_check = corrected_labels.merge(
    saved_decisions[
        [
            "record_id",
            "original_llm_label",
            "corrected_label",
            "correction_applied"
        ]
    ],
    on="record_id",
    validate="one_to_one",
    suffixes=("_labels", "_decisions")
)

for column in [
    "original_llm_label",
    "corrected_label",
    "correction_applied"
]:
    assert (
        decision_check[f"{column}_labels"]
        == decision_check[f"{column}_decisions"]
    ).all()

# Restore frozen original text and LLM labels.
source = (
    training_df[["record_id", "text", "llm_label"]]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(source) == 13621
assert source["record_id"].is_unique
assert source["text"].notna().all()

assert text_fingerprint(
    source[["record_id", "text"]]
) == training_spec["raw_text_sha256"]

combined = source.merge(
    corrected_labels,
    on="record_id",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(combined) == 13621
assert set(combined["record_id"].astype(int)) == training_ids

assert (
    combined["llm_label"]
    == combined["original_llm_label"]
).all()

original = combined["llm_label"].to_numpy(dtype=int)
corrected = combined["corrected_label"].to_numpy(dtype=int)
changed = combined["correction_applied"].to_numpy(dtype=bool)

assert np.isin(original, range(4)).all()
assert np.isin(corrected, range(4)).all()
assert np.array_equal(changed, original != corrected)
assert int(changed.sum()) == 705

# Independently reconstruct decisions from frozen neighborhood evidence.
evidence_check = combined[
    ["record_id", "llm_label"]
].merge(
    evidence[
        ["record_id", "neighbor_top_label"]
        + [f"support_class_{i}" for i in range(4)]
    ],
    on="record_id",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

support = evidence_check[
    [f"support_class_{i}" for i in range(4)]
].to_numpy(dtype=float)

neighbor = evidence_check[
    "neighbor_top_label"
].to_numpy(dtype=int)

assert np.isfinite(support).all()
assert np.allclose(support.sum(axis=1), 1.0, atol=1e-6)
assert np.array_equal(support.argmax(axis=1), neighbor)

top_support = support.max(axis=1)
llm_support = support[np.arange(len(original)), original]

expected_changed = (
    (original != neighbor)
    & (top_support >= frozen_rule["min_neighbor_support"])
    & (
        top_support - llm_support
        >= frozen_rule["min_semantic_support_gap"]
    )
)

assert np.array_equal(changed, expected_changed)
assert np.array_equal(
    corrected,
    np.where(expected_changed, neighbor, original)
)

# Prepare matched datasets using generated training targets.
baseline_df = combined[
    ["record_id", "text"]
].copy()

baseline_df["label"] = original

corrected_df = combined[
    ["record_id", "text"]
].copy()

corrected_df["label"] = corrected

assert len(baseline_df) == len(corrected_df) == 13621

assert baseline_df["record_id"].equals(
    corrected_df["record_id"]
)

assert baseline_df["text"].equals(
    corrected_df["text"]
)

assert int(
    (baseline_df["label"] != corrected_df["label"]).sum()
) == 705

# Save the two datasets without modifying source artifacts.
BASELINE_PATH = (
    DOWNSTREAM_DIR / "baseline_training_v1.csv"
)

CORRECTED_PATH = (
    DOWNSTREAM_DIR / "corrected_training_v1.csv"
)

baseline_status = save_immutable(
    BASELINE_PATH,
    baseline_df.to_csv(
        index=False,
        lineterminator="\n"
    )
)

corrected_status = save_immutable(
    CORRECTED_PATH,
    corrected_df.to_csv(
        index=False,
        lineterminator="\n"
    )
)

# Record precise provenance for reproducibility.
dataset_manifest = {
    "protocol_version": "v1",
    "experiment": "MATCHED_DOWNSTREAM_TRAINING_DATA",
    "records_per_condition": 13621,
    "conditions": [
        "ORIGINAL_LLM_LABELS",
        "FROZEN_S_SLTE_CORRECTED_LABELS"
    ],
    "text_policy": "IDENTICAL_FROZEN_ORIGINAL_TEXT",
    "record_order": "ASCENDING_RECORD_ID",
    "training_target_semantics": (
        "Generated labels; not trusted reference labels"
    ),
    "label_changes": 705,
    "class_count_original": {
        str(i): int((original == i).sum())
        for i in range(4)
    },
    "class_count_corrected": {
        str(i): int((corrected == i).sum())
        for i in range(4)
    },
    "source_sha256": {
        "partition_manifest": file_digest(
            BASE / "configs/holdout_partition_manifest_v1.json"
        ),
        "correction_manifest": file_digest(
            CORRECTION_MANIFEST_PATH
        ),
        "frozen_rule": file_digest(RULE_PATH)
    },
    "output_sha256": {
        "baseline_training": file_digest(BASELINE_PATH),
        "corrected_training": file_digest(CORRECTED_PATH)
    },
    "reference_labels_accessed": False,
    "holdout_accessed": False,
    "downstream_training_started": False
}

DATASET_MANIFEST_PATH = (
    DOWNSTREAM_DIR / "matched_training_manifest_v1.json"
)

manifest_status = save_immutable(
    DATASET_MANIFEST_PATH,
    json.dumps(dataset_manifest, indent=2) + "\n"
)

print("STAGE 13 — MATCHED TRAINING DATASETS")

print("\nDATASET VERIFICATION")
print("Baseline records:", len(baseline_df))
print("Corrected records:", len(corrected_df))
print("Identical record IDs: YES")
print("Identical original texts: YES")
print("Changed labels:", int(changed.sum()))

print("\nTRAINING CLASS DISTRIBUTION")
display(pd.DataFrame([
    {
        "class": name,
        "baseline": int((original == i).sum()),
        "corrected": int((corrected == i).sum())
    }
    for i, name in enumerate(
        ["World", "Sports", "Business", "Sci/Tech"]
    )
]))

print("\nARTIFACTS")
print("Baseline CSV:", baseline_status)
print("Corrected CSV:", corrected_status)
print("Manifest:", manifest_status)
print("Directory:", DOWNSTREAM_DIR)

print("\nRESEARCH INTEGRITY")
print("Frozen rule verified: YES")
print("Correction decisions independently reproduced: YES")
print("Training reference labels accessed: NO")
print("Holdout accessed: NO")
print("Model training started: NO")

print("\nSTATUS: MATCHED TRAINING DATASETS VERIFIED")


STAGE 13 — MATCHED TRAINING DATASETS

DATASET VERIFICATION
Baseline records: 13621
Corrected records: 13621
Identical record IDs: YES
Identical original texts: YES
Changed labels: 705

TRAINING CLASS DISTRIBUTION


,class,baseline,corrected
0,World,4199,4008
1,Sports,3532,3542
2,Business,4015,3662
3,Sci/Tech,1875,2409



ARTIFACTS
Baseline CSV: NEW_SAVED
Corrected CSV: NEW_SAVED
Manifest: NEW_SAVED
Directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1

RESEARCH INTEGRITY
Frozen rule verified: YES
Correction decisions independently reproduced: YES
Training reference labels accessed: NO
Holdout accessed: NO
Model training started: NO

STATUS: MATCHED TRAINING DATASETS VERIFIED


In [ ]:

%pip install -q "huggingface_hub==1.33.0"

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import HfApi

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
DOWNSTREAM_DIR = BASE / "outputs/s_slte_downstream_v1"

BASELINE_PATH = DOWNSTREAM_DIR / "baseline_training_v1.csv"
CORRECTED_PATH = DOWNSTREAM_DIR / "corrected_training_v1.csv"
DATASET_MANIFEST_PATH = (
    DOWNSTREAM_DIR / "matched_training_manifest_v1.json"
)
PROTOCOL_PATH = (
    DOWNSTREAM_DIR / "downstream_training_protocol_v1.json"
)

MODEL_ID = "distilbert/distilbert-base-uncased"
SEEDS = [42, 123, 2026]

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode())
        digest.update(b"\0")
        digest.update(str(row.text).encode())
        digest.update(b"\0")

    return digest.hexdigest()

# Verify the frozen training datasets.
with DATASET_MANIFEST_PATH.open(encoding="utf-8") as f:
    dataset_manifest = json.load(f)

assert dataset_manifest["records_per_condition"] == 13621
assert dataset_manifest["label_changes"] == 705

assert file_hash(BASELINE_PATH) == (
    dataset_manifest["output_sha256"]["baseline_training"]
)
assert file_hash(CORRECTED_PATH) == (
    dataset_manifest["output_sha256"]["corrected_training"]
)

baseline = pd.read_csv(BASELINE_PATH)
corrected = pd.read_csv(CORRECTED_PATH)

assert len(baseline) == len(corrected) == 13621
assert baseline["record_id"].is_unique
assert corrected["record_id"].is_unique

assert baseline["record_id"].equals(corrected["record_id"])
assert baseline["text"].equals(corrected["text"])

assert baseline["label"].isin(range(4)).all()
assert corrected["label"].isin(range(4)).all()

changes = int(
    (baseline["label"] != corrected["label"]).sum()
)
assert changes == 705

# Match the exact frozen active-training text.
with (
    BASE / "configs/holdout_partition_manifest_v1.json"
).open(encoding="utf-8") as f:
    partition_manifest = json.load(f)

training_spec = partition_manifest["partitions"]["training"]

assert text_fingerprint(
    baseline[["record_id", "text"]]
) == training_spec["raw_text_sha256"]

# Pin the pretrained model revision.
if PROTOCOL_PATH.exists():
    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        previous_protocol = json.load(f)

    model_revision = previous_protocol["model_revision_sha"]
else:
    model_revision = HfApi().model_info(MODEL_ID).sha

assert isinstance(model_revision, str)
assert len(model_revision) == 40
assert all(c in "0123456789abcdef" for c in model_revision)

protocol = {
    "protocol_version": "v1",
    "experiment": "S_SLTE_MATCHED_DOWNSTREAM_COMPARISON",
    "model_id": MODEL_ID,
    "model_revision_sha": model_revision,
    "model_task": "SEQUENCE_CLASSIFICATION",
    "num_labels": 4,
    "class_names": [
        "World", "Sports", "Business", "Sci/Tech"
    ],
    "conditions": {
        "baseline": BASELINE_PATH.name,
        "corrected": CORRECTED_PATH.name
    },
    "seeds": SEEDS,
    "training": {
        "epochs": 2,
        "train_batch_size": 16,
        "gradient_accumulation_steps": 2,
        "learning_rate": 2e-5,
        "optimizer": "AdamW",
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "lr_scheduler": "linear",
        "max_length": 256,
        "dynamic_padding": True,
        "drop_last": False,
        "class_weights": None,
        "early_stopping": False,
        "load_best_model_at_end": False,
        "final_checkpoint_policy": "LAST_EPOCH"
    },
    "matched_controls": {
        "same_pretrained_revision": True,
        "same_training_record_ids": True,
        "same_training_text": True,
        "same_record_order": True,
        "same_seed_per_pair": True,
        "reset_seed_before_model_initialization": True,
        "same_batch_order_per_pair": True,
        "same_optimizer_and_schedule": True,
        "only_training_labels_differ": True
    },
    "evaluation_plan": {
        "primary_metric": "holdout_macro_f1",
        "secondary_metrics": [
            "holdout_accuracy",
            "per_class_precision_recall_f1"
        ],
        "report_per_seed": True,
        "report_mean_and_standard_deviation": True,
        "paired_comparison": True,
        "holdout_evaluation_timing": (
            "AFTER_ALL_SIX_TRAINING_RUNS"
        ),
        "holdout_used_for_tuning": False
    },
    "input_sha256": {
        "baseline_csv": file_hash(BASELINE_PATH),
        "corrected_csv": file_hash(CORRECTED_PATH),
        "matched_dataset_manifest": file_hash(
            DATASET_MANIFEST_PATH
        )
    },
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "training_started": False,
    "evaluation_started": False
}

content = json.dumps(protocol, indent=2) + "\n"

if PROTOCOL_PATH.exists():
    assert PROTOCOL_PATH.read_text(
        encoding="utf-8"
    ) == content
    status = "EXISTING_VERIFIED"
else:
    temporary = PROTOCOL_PATH.with_suffix(".json.tmp")
    temporary.write_text(content, encoding="utf-8")
    os.replace(temporary, PROTOCOL_PATH)
    status = "NEW_SAVED"

print("STAGE 13 — DOWNSTREAM TRAINING PROTOCOL")
print("Model:", MODEL_ID)
print("Model revision:", model_revision)
print("Training records per condition:", len(baseline))
print("Changed labels:", changes)
print("Random seeds:", SEEDS)
print("Epochs:", protocol["training"]["epochs"])
print("Total planned training runs:", 2 * len(SEEDS))
print("Learning rate:", protocol["training"]["learning_rate"])
print("Maximum length:", protocol["training"]["max_length"])

print("\nREPRODUCIBILITY")
print("Training text fingerprint: VERIFIED")
print("Training dataset fingerprints: VERIFIED")
print("Model revision: PINNED")
print("Protocol:", status)
print("Saved path:", PROTOCOL_PATH)

print("\nRESEARCH INTEGRITY")
print("Training labels are generated targets: YES")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Training started: NO")
print("Evaluation started: NO")

print("\nSTATUS: DOWNSTREAM TRAINING PROTOCOL FROZEN")


STAGE 13 — DOWNSTREAM TRAINING PROTOCOL
Model: distilbert/distilbert-base-uncased
Model revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
Training records per condition: 13621
Changed labels: 705
Random seeds: [42, 123, 2026]
Epochs: 2
Total planned training runs: 6
Learning rate: 2e-05
Maximum length: 256

REPRODUCIBILITY
Training text fingerprint: VERIFIED
Training dataset fingerprints: VERIFIED
Model revision: PINNED
Protocol: EXISTING_VERIFIED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/downstream_training_protocol_v1.json

RESEARCH INTEGRITY
Training labels are generated targets: YES
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Training started: NO
Evaluation started: NO

STATUS: DOWNSTREAM TRAINING PROTOCOL FROZEN


In [ ]:

from google.colab import drive
from pathlib import Path
import json

# Restore Google Drive after a runtime change.
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

PROTOCOL_PATH = (
    BASE / "outputs/s_slte_downstream_v1"
    / "downstream_training_protocol_v1.json"
)

print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Research workspace exists:", BASE.is_dir())
print("Protocol exists:", PROTOCOL_PATH.is_file())

if PROTOCOL_PATH.is_file():
    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        protocol = json.load(f)

    assert protocol["model_id"] == (
        "distilbert/distilbert-base-uncased"
    )
    assert protocol["training"]["epochs"] == 2
    assert protocol["seeds"] == [42, 123, 2026]

    print("\nModel:", protocol["model_id"])
    print("Revision:", protocol["model_revision_sha"])
    print("Training runs:", 2 * len(protocol["seeds"]))
    print("\nSTATUS: TRAINING PROTOCOL RESTORED")

else:
    print("\nProtocol not found at:", PROTOCOL_PATH)

    if BASE.is_dir():
        output_dir = BASE / "outputs"
        print(
            "Downstream folder exists:",
            (output_dir / "s_slte_downstream_v1").is_dir()
        )
    else:
        print("Check the mounted Google account and workspace path.")

    print("\nSTATUS: DRIVE PATH NEEDS INVESTIGATION")


Mounted at /content/drive
Drive mounted: True
Research workspace exists: True
Protocol exists: True

Model: distilbert/distilbert-base-uncased
Revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
Training runs: 6

STATUS: TRAINING PROTOCOL RESTORED


In [ ]:

%pip install -q "transformers==5.20.1" "accelerate" "safetensors"

import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import json
import random
import hashlib
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
DOWNSTREAM_DIR = BASE / "outputs/s_slte_downstream_v1"

PROTOCOL_PATH = (
    DOWNSTREAM_DIR / "downstream_training_protocol_v1.json"
)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert PROTOCOL_PATH.is_file()

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    protocol = json.load(f)

MODEL_ID = protocol["model_id"]
MODEL_REVISION = protocol["model_revision_sha"]

assert MODEL_ID == "distilbert/distilbert-base-uncased"
assert MODEL_REVISION == (
    "12040accade4e8a0f71eabdb258fecc2e7e948be"
)
assert protocol["num_labels"] == 4
assert protocol["seeds"] == [42, 123, 2026]
assert protocol["training"]["epochs"] == 2
assert transformers.__version__ == "5.20.1"

assert torch.cuda.is_available(), "Enable T4 GPU first."

DEVICE = torch.device("cuda:0")

# Reproducible smoke-test initialization.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Load exact pinned public checkpoint.
downstream_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=False,
    use_fast=True
)

downstream_model = (
    AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        num_labels=4
    )
)

downstream_model.to(DEVICE)
downstream_model.eval()

assert downstream_model.config.num_labels == 4
assert downstream_tokenizer.is_fast
assert next(downstream_model.parameters()).device.type == "cuda"

# Verify both frozen training datasets.
BASELINE_PATH = DOWNSTREAM_DIR / "baseline_training_v1.csv"
CORRECTED_PATH = DOWNSTREAM_DIR / "corrected_training_v1.csv"

assert file_hash(BASELINE_PATH) == (
    protocol["input_sha256"]["baseline_csv"]
)

assert file_hash(CORRECTED_PATH) == (
    protocol["input_sha256"]["corrected_csv"]
)

sample = pd.read_csv(BASELINE_PATH, nrows=4)

assert len(sample) == 4
assert sample["text"].notna().all()

inputs = downstream_tokenizer(
    sample["text"].astype(str).tolist(),
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

assert inputs["input_ids"].shape[0] == 4
assert inputs["input_ids"].shape[1] <= 256

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

# Forward pass only; no optimizer updates.
with torch.inference_mode():
    logits = downstream_model(**inputs).logits

assert logits.shape == (4, 4)
assert torch.isfinite(logits).all()

print("STAGE 13 — DISTILBERT READINESS TEST")

print("\nMODEL")
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Accelerate:", metadata.version("accelerate"))

print("\nGPU")
print("Device:", torch.cuda.get_device_name(0))
print(
    "Allocated VRAM:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nVALIDATION")
print("Classes:", downstream_model.config.num_labels)
print("Tokenizer:", type(downstream_tokenizer).__name__)
print("Forward-pass shape:", tuple(logits.shape))
print("Finite logits: YES")
print("Training datasets: VERIFIED")
print("Pinned checkpoint: VERIFIED")

print("\nRESEARCH INTEGRITY")
print("Training performed: NO")
print("Optimizer updates: NO")
print("Training reference labels accessed: NO")
print("Holdout accessed: NO")

print("\nSTATUS: DOWNSTREAM MODEL READY FOR TRAINING")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — DISTILBERT READINESS TEST

MODEL
Model: distilbert/distilbert-base-uncased
Revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
Transformers: 5.20.1
PyTorch: 2.14.1+cu130
Accelerate: 1.15.0

GPU
Device: Tesla T4
Allocated VRAM: 0.26 GB

VALIDATION
Classes: 4
Tokenizer: BertTokenizer
Forward-pass shape: (4, 4)
Finite logits: YES
Training datasets: VERIFIED
Pinned checkpoint: VERIFIED

RESEARCH INTEGRITY
Training performed: NO
Optimizer updates: NO
Training reference labels accessed: NO
Holdout accessed: NO

STATUS: DOWNSTREAM MODEL READY FOR TRAINING


In [ ]:

import os
import gc
import json
import math
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

with (ROOT / "downstream_training_protocol_v1.json").open(
    encoding="utf-8"
) as f:
    cfg = json.load(f)

SEED = 42
CONDITION = "baseline"

assert SEED in cfg["seeds"]
assert cfg["model_id"] == "distilbert/distilbert-base-uncased"
assert cfg["training"]["epochs"] == 2
assert cfg["training"]["train_batch_size"] == 16
assert cfg["training"]["gradient_accumulation_steps"] == 2
assert cfg["training"]["final_checkpoint_policy"] == "LAST_EPOCH"
assert cfg["training"]["early_stopping"] is False
assert cfg["training"]["drop_last"] is False
assert torch.cuda.is_available()

DEVICE = torch.device("cuda:0")

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

protocol_path = ROOT / "downstream_training_protocol_v1.json"
data_path = ROOT / "baseline_training_v1.csv"
paired_path = ROOT / "corrected_training_v1.csv"

assert sha256(data_path) == cfg["input_sha256"]["baseline_csv"]
assert sha256(paired_path) == cfg["input_sha256"]["corrected_csv"]

data = pd.read_csv(data_path)
paired = pd.read_csv(paired_path)

assert len(data) == len(paired) == 13621
assert data["record_id"].is_unique
assert data["record_id"].equals(paired["record_id"])
assert data["text"].equals(paired["text"])
assert data["text"].notna().all()
assert data["label"].isin(range(4)).all()
assert int((data["label"] != paired["label"]).sum()) == 705

texts = data["text"].astype(str).tolist()
labels = data["label"].to_numpy(dtype=np.int64)

batch_size = cfg["training"]["train_batch_size"]
accumulation = cfg["training"]["gradient_accumulation_steps"]
epochs = cfg["training"]["epochs"]

batches_per_epoch = math.ceil(len(data) / batch_size)
updates_per_epoch = math.ceil(batches_per_epoch / accumulation)
total_updates = updates_per_epoch * epochs

warmup_steps = math.ceil(
    total_updates * cfg["training"]["warmup_ratio"]
)

RUN_DIR = ROOT / "training_runs" / f"{CONDITION}_seed_{SEED}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = RUN_DIR / "training_checkpoint.pt"
model_dir = RUN_DIR / "final_model"
summary_path = RUN_DIR / "training_summary_v1.json"

if summary_path.exists():
    with summary_path.open(encoding="utf-8") as f:
        completed = json.load(f)

    assert completed["status"] == "TRAINING_COMPLETED"
    assert completed["seed"] == SEED
    assert completed["condition"] == CONDITION
    assert completed["protocol_sha256"] == sha256(protocol_path)
    assert completed["training_csv_sha256"] == sha256(data_path)
    assert (model_dir / "model.safetensors").exists()
    assert completed["model_sha256"] == sha256(
        model_dir / "model.safetensors"
    )

    print("STAGE 13 — BASELINE SEED 42")
    print("Existing training run: VERIFIED")
    print("Optimizer updates:", completed["optimizer_updates"])
    print("Epoch losses:", completed["epoch_losses"])
    print("STATUS: BASELINE SEED 42 ALREADY COMPLETED")

else:
    # Remove the earlier smoke-test model to reclaim GPU memory.
    if "downstream_model" in globals():
        del downstream_model

    gc.collect()
    torch.cuda.empty_cache()

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.benchmark = False

    tokenizer = AutoTokenizer.from_pretrained(
        cfg["model_id"],
        revision=cfg["model_revision_sha"],
        token=False,
        use_fast=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_id"],
        revision=cfg["model_revision_sha"],
        token=False,
        num_labels=4
    ).to(DEVICE)

    # Same parameter grouping will be used for both conditions.
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")

    optimizer = torch.optim.AdamW(
        [
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if p.requires_grad
                    and not any(x in n for x in no_decay)
                ],
                "weight_decay": cfg["training"]["weight_decay"]
            },
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if p.requires_grad
                    and any(x in n for x in no_decay)
                ],
                "weight_decay": 0.0
            }
        ],
        lr=cfg["training"]["learning_rate"],
        foreach=False
    )

    def lr_multiplier(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        return max(
            0.0,
            (total_updates - step)
            / max(1, total_updates - warmup_steps)
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_multiplier
    )

    start_epoch = 0
    start_batch = 0
    global_step = 0

    epoch_losses = []
    loss_sum = 0.0
    examples_seen = 0

    if checkpoint_path.exists():
        state = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        assert state["seed"] == SEED
        assert state["condition"] == CONDITION
        assert state["protocol_sha256"] == sha256(protocol_path)
        assert state["training_csv_sha256"] == sha256(data_path)

        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["global_step"]
        epoch_losses = state["epoch_losses"]
        loss_sum = state["loss_sum"]
        examples_seen = state["examples_seen"]

        torch.set_rng_state(state["cpu_rng_state"])
        torch.cuda.set_rng_state_all(
            state["cuda_rng_states"]
        )

        print(
            "RESUMED:",
            f"epoch={start_epoch + 1}",
            f"batch={start_batch}",
            f"updates={global_step}"
        )

    def save_checkpoint(epoch, next_batch):
        state = {
            "seed": SEED,
            "condition": CONDITION,
            "protocol_sha256": sha256(protocol_path),
            "training_csv_sha256": sha256(data_path),
            "epoch": epoch,
            "next_batch": next_batch,
            "global_step": global_step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "cpu_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all(),
            "epoch_losses": epoch_losses,
            "loss_sum": loss_sum,
            "examples_seen": examples_seen
        }

        temporary = checkpoint_path.with_suffix(".pt.tmp")
        torch.save(state, temporary)
        os.replace(temporary, checkpoint_path)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    print("STAGE 13 — BASELINE TRAINING STARTED")
    print("Condition:", CONDITION)
    print("Seed:", SEED)
    print("Training records:", len(data))
    print("Epochs:", epochs)
    print("Batches per epoch:", batches_per_epoch)
    print("Total optimizer updates:", total_updates)
    print("Warmup updates:", warmup_steps)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Holdout accessed: NO")

    for epoch in range(start_epoch, epochs):
        permutation = np.random.default_rng(
            SEED + epoch
        ).permutation(len(data))

        batch_begin = start_batch if epoch == start_epoch else 0

        for batch_index in range(batch_begin, batches_per_epoch):
            start = batch_index * batch_size
            end = min(start + batch_size, len(data))

            positions = permutation[start:end]

            encoded = tokenizer(
                [texts[int(i)] for i in positions],
                padding=True,
                truncation=True,
                max_length=cfg["training"]["max_length"],
                return_tensors="pt"
            )

            encoded = {
                k: v.to(DEVICE)
                for k, v in encoded.items()
            }

            targets = torch.tensor(
                labels[positions],
                dtype=torch.long,
                device=DEVICE
            )

            outputs = model(
                **encoded,
                labels=targets
            )

            loss = outputs.loss
            assert torch.isfinite(loss).item()

            loss_sum += loss.item() * len(positions)
            examples_seen += len(positions)

            remaining_in_group = (
                batches_per_epoch
                - (batch_index // accumulation) * accumulation
            )

            group_size = min(accumulation, remaining_in_group)

            (loss / group_size).backward()

            del outputs, loss, encoded, targets

            update_due = (
                (batch_index + 1) % accumulation == 0
                or batch_index + 1 == batches_per_epoch
            )

            if update_due:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                if global_step % 100 == 0:
                    print(
                        f"Epoch {epoch + 1}/{epochs} | "
                        f"Optimizer updates {global_step}/{total_updates}"
                    )

                    save_checkpoint(epoch, batch_index + 1)

        epoch_loss = loss_sum / examples_seen
        epoch_losses.append(round(float(epoch_loss), 6))

        print(
            f"EPOCH {epoch + 1} COMPLETED | "
            f"Mean training loss: {epoch_loss:.6f}"
        )

        loss_sum = 0.0
        examples_seen = 0

        save_checkpoint(epoch + 1, 0)

    assert global_step == total_updates
    assert len(epoch_losses) == epochs

    model.eval()
    model_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        model_dir,
        safe_serialization=True
    )
    tokenizer.save_pretrained(model_dir)

    weight_path = model_dir / "model.safetensors"
    assert weight_path.is_file()

    summary = {
        "status": "TRAINING_COMPLETED",
        "condition": CONDITION,
        "seed": SEED,
        "records": len(data),
        "epochs": epochs,
        "optimizer_updates": global_step,
        "warmup_updates": warmup_steps,
        "epoch_losses": epoch_losses,
        "model_id": cfg["model_id"],
        "model_revision_sha": cfg["model_revision_sha"],
        "protocol_sha256": sha256(protocol_path),
        "training_csv_sha256": sha256(data_path),
        "model_sha256": sha256(weight_path),
        "precision": "FLOAT32",
        "gradient_clipping": "NONE",
        "batch_order": "NUMPY_DEFAULT_RNG_SEED_PLUS_EPOCH",
        "optimizer": "ADAMW_DECOUPLED_WEIGHT_DECAY",
        "weight_decay_exclusions": list(no_decay),
        "evaluation_performed": False,
        "holdout_accessed": False,
        "reference_labels_accessed": False
    }

    temporary = summary_path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(summary, indent=2) + "\n",
        encoding="utf-8"
    )
    os.replace(temporary, summary_path)

    assert summary_path.is_file()

    # The verified final model replaces the resumable checkpoint.
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    print("\nSTAGE 13 — BASELINE SEED 42 COMPLETED")
    print("Training records:", len(data))
    print("Epoch losses:", epoch_losses)
    print("Optimizer updates:", global_step)
    print("Model:", model_dir)
    print("Summary:", summary_path)
    print("Training reference labels accessed: NO")
    print("Holdout accessed: NO")
    print("Evaluation performed: NO")
    print("\nSTATUS: BASELINE SEED 42 TRAINING PASSED")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — BASELINE TRAINING STARTED
Condition: baseline
Seed: 42
Training records: 13621
Epochs: 2
Batches per epoch: 852
Total optimizer updates: 852
Warmup updates: 52
GPU: Tesla T4
Holdout accessed: NO
Epoch 1/2 | Optimizer updates 100/852
Epoch 1/2 | Optimizer updates 200/852
Epoch 1/2 | Optimizer updates 300/852
Epoch 1/2 | Optimizer updates 400/852
EPOCH 1 COMPLETED | Mean training loss: 0.451491
Epoch 2/2 | Optimizer updates 500/852
Epoch 2/2 | Optimizer updates 600/852
Epoch 2/2 | Optimizer updates 700/852
Epoch 2/2 | Optimizer updates 800/852
EPOCH 2 COMPLETED | Mean training loss: 0.230441


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — BASELINE SEED 42 COMPLETED
Training records: 13621
Epoch losses: [0.451491, 0.230441]
Optimizer updates: 852
Model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_42/final_model
Summary: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_42/training_summary_v1.json
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: BASELINE SEED 42 TRAINING PASSED


In [ ]:

import os
import gc
import json
import math
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

SEED = 42
CONDITION = "corrected"

PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"
DATA_PATH = ROOT / "corrected_training_v1.csv"
PAIRED_PATH = ROOT / "baseline_training_v1.csv"

RUN_DIR = ROOT / "training_runs" / f"{CONDITION}_seed_{SEED}"
CHECKPOINT_PATH = RUN_DIR / "training_checkpoint.pt"
MODEL_DIR = RUN_DIR / "final_model"
SUMMARY_PATH = RUN_DIR / "training_summary_v1.json"

LOCAL_DIR = Path("/content/s_slte_corrected_seed42")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def run_corrected_training():

    assert PROTOCOL_PATH.is_file()
    assert torch.cuda.is_available(), "Enable T4 GPU."

    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        cfg = json.load(f)

    assert cfg["model_id"] == "distilbert/distilbert-base-uncased"
    assert cfg["model_revision_sha"] == (
        "12040accade4e8a0f71eabdb258fecc2e7e948be"
    )
    assert SEED in cfg["seeds"]
    assert cfg["num_labels"] == 4

    settings = cfg["training"]

    assert settings["epochs"] == 2
    assert settings["train_batch_size"] == 16
    assert settings["gradient_accumulation_steps"] == 2
    assert settings["learning_rate"] == 2e-5
    assert settings["optimizer"] == "AdamW"
    assert settings["lr_scheduler"] == "linear"
    assert settings["early_stopping"] is False
    assert settings["drop_last"] is False

    protocol_hash = sha256_file(PROTOCOL_PATH)

    # Copy frozen inputs to local storage before training.
    local_data = LOCAL_DIR / DATA_PATH.name
    local_paired = LOCAL_DIR / PAIRED_PATH.name

    shutil.copyfile(DATA_PATH, local_data)
    shutil.copyfile(PAIRED_PATH, local_paired)

    data_hash = sha256_file(local_data)
    paired_hash = sha256_file(local_paired)

    assert data_hash == cfg["input_sha256"]["corrected_csv"]
    assert paired_hash == cfg["input_sha256"]["baseline_csv"]

    if SUMMARY_PATH.exists():
        with SUMMARY_PATH.open(encoding="utf-8") as f:
            completed = json.load(f)

        assert completed["status"] == "TRAINING_COMPLETED"
        assert completed["condition"] == CONDITION
        assert completed["seed"] == SEED
        assert completed["protocol_sha256"] == protocol_hash
        assert completed["training_csv_sha256"] == data_hash
        assert completed["optimizer_updates"] == 852

        weights = MODEL_DIR / "model.safetensors"
        assert weights.is_file()
        assert sha256_file(weights) == completed["model_sha256"]

        print("STAGE 13 — CORRECTED SEED 42")
        print("Existing completed run: VERIFIED")
        print("Epoch losses:", completed["epoch_losses"])
        print("Optimizer updates:", completed["optimizer_updates"])
        print("\nSTATUS: CORRECTED SEED 42 TRAINING PASSED")
        return

    data = pd.read_csv(local_data)
    paired = pd.read_csv(local_paired)

    assert len(data) == len(paired) == 13621
    assert data["record_id"].is_unique
    assert data["record_id"].equals(paired["record_id"])
    assert data["text"].equals(paired["text"])
    assert data["text"].notna().all()
    assert data["label"].isin(range(4)).all()
    assert paired["label"].isin(range(4)).all()

    changed = int(
        (data["label"] != paired["label"]).sum()
    )

    assert changed == 705

    texts = data["text"].astype(str).tolist()
    labels = data["label"].to_numpy(dtype=np.int64)

    batch_size = settings["train_batch_size"]
    accumulation = settings["gradient_accumulation_steps"]
    epochs = settings["epochs"]

    batches_per_epoch = math.ceil(len(data) / batch_size)
    updates_per_epoch = math.ceil(
        batches_per_epoch / accumulation
    )
    total_updates = updates_per_epoch * epochs

    warmup_steps = math.ceil(
        total_updates * settings["warmup_ratio"]
    )

    assert batches_per_epoch == 852
    assert total_updates == 852
    assert warmup_steps == 52

    device = torch.device("cuda:0")

    # The previous cell may still hold the interrupted model.
    old = globals()

    memory_names = [
        "model", "optimizer", "scheduler", "tokenizer",
        "global_step", "epoch", "batch_index",
        "epoch_losses", "loss_sum", "examples_seen",
        "labels", "texts"
    ]

    memory_available = (
        all(name in old for name in memory_names)
        and old.get("CONDITION") == CONDITION
        and old.get("SEED") == SEED
    )

    if memory_available:
        memory_available = (
            isinstance(old["model"], torch.nn.Module)
            and isinstance(
                old["optimizer"], torch.optim.Optimizer
            )
            and 0 < int(old["global_step"]) < total_updates
            and int(old["epoch"]) in (0, 1)
            and 0 <= int(old["batch_index"]) < batches_per_epoch
            and np.array_equal(old["labels"], labels)
            and old["texts"] == texts
            and old["model"].training
            and next(old["model"].parameters()).device.type == "cuda"
        )

    if memory_available:
        memory_available = all(
            parameter.grad is None
            for parameter in old["model"].parameters()
        )

    if memory_available:
        old_epoch = int(old["epoch"])
        old_next_batch = int(old["batch_index"]) + 1

        expected_step = (
            old_epoch * updates_per_epoch
            + math.ceil(old_next_batch / accumulation)
        )

        memory_available = (
            old_next_batch < batches_per_epoch
            and old_next_batch % accumulation == 0
            and int(old["global_step"]) == expected_step
            and len(old["epoch_losses"]) == old_epoch
        )

    recovery_mode = "FRESH"

    if memory_available:
        model = old["model"]
        optimizer = old["optimizer"]
        scheduler = old["scheduler"]
        tokenizer = old["tokenizer"]

        start_epoch = int(old["epoch"])
        start_batch = int(old["batch_index"]) + 1
        global_step = int(old["global_step"])

        epoch_losses = list(old["epoch_losses"])
        loss_sum = float(old["loss_sum"])
        examples_seen = int(old["examples_seen"])

        recovery_mode = "IN_MEMORY"

    else:
        # A new initialization uses the same frozen seed.
        gc.collect()
        torch.cuda.empty_cache()

        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = False

        tokenizer = AutoTokenizer.from_pretrained(
            cfg["model_id"],
            revision=cfg["model_revision_sha"],
            token=False,
            use_fast=True
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            cfg["model_id"],
            revision=cfg["model_revision_sha"],
            token=False,
            num_labels=4
        ).to(device)

        no_decay = (
            "bias", "LayerNorm.weight", "layer_norm.weight"
        )

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": [
                        p for n, p in model.named_parameters()
                        if p.requires_grad
                        and not any(x in n for x in no_decay)
                    ],
                    "weight_decay": settings["weight_decay"]
                },
                {
                    "params": [
                        p for n, p in model.named_parameters()
                        if p.requires_grad
                        and any(x in n for x in no_decay)
                    ],
                    "weight_decay": 0.0
                }
            ],
            lr=settings["learning_rate"],
            foreach=False
        )

        def lr_multiplier(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)

            return max(
                0.0,
                (total_updates - step)
                / max(1, total_updates - warmup_steps)
            )

        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lr_multiplier
        )

        start_epoch = 0
        start_batch = 0
        global_step = 0

        epoch_losses = []
        loss_sum = 0.0
        examples_seen = 0

        # Restore the last completed atomic checkpoint.
        if CHECKPOINT_PATH.exists():
            state = torch.load(
                CHECKPOINT_PATH,
                map_location="cpu",
                weights_only=True
            )

            assert state["condition"] == CONDITION
            assert state["seed"] == SEED
            assert state["protocol_sha256"] == protocol_hash
            assert state["training_csv_sha256"] == data_hash

            model.load_state_dict(state["model"])
            optimizer.load_state_dict(state["optimizer"])
            scheduler.load_state_dict(state["scheduler"])

            start_epoch = int(state["epoch"])
            start_batch = int(state["next_batch"])
            global_step = int(state["global_step"])

            epoch_losses = list(state["epoch_losses"])
            loss_sum = float(state["loss_sum"])
            examples_seen = int(state["examples_seen"])

            assert 0 <= start_epoch <= epochs
            assert 0 <= start_batch <= batches_per_epoch
            assert global_step <= total_updates

            torch.set_rng_state(state["cpu_rng_state"])
            torch.cuda.set_rng_state_all(
                state["cuda_rng_states"]
            )

            recovery_mode = "DISK_CHECKPOINT"

    # Preserve exact training order for the matched pair.
    model.train()
    optimizer.zero_grad(set_to_none=True)

    def save_checkpoint(next_epoch, next_batch):
        state = {
            "condition": CONDITION,
            "seed": SEED,
            "protocol_sha256": protocol_hash,
            "training_csv_sha256": data_hash,
            "epoch": next_epoch,
            "next_batch": next_batch,
            "global_step": global_step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "cpu_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all(),
            "epoch_losses": epoch_losses,
            "loss_sum": loss_sum,
            "examples_seen": examples_seen
        }

        # Serialize locally; synchronize to Drive atomically.
        local_temp = LOCAL_DIR / "training_checkpoint.pt"
        drive_temp = CHECKPOINT_PATH.with_suffix(".pt.tmp")

        torch.save(state, local_temp)
        shutil.copyfile(local_temp, drive_temp)
        os.replace(drive_temp, CHECKPOINT_PATH)

        local_temp.unlink(missing_ok=True)

    print("STAGE 13 — S-SLTE CORRECTED TRAINING")
    print("Condition:", CONDITION)
    print("Seed:", SEED)
    print("Recovery mode:", recovery_mode)
    print("Starting optimizer update:", global_step)
    print("Total optimizer updates:", total_updates)
    print("Training records:", len(data))
    print("Changed labels:", changed)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Holdout accessed: NO")

    for epoch in range(start_epoch, epochs):
        permutation = np.random.default_rng(
            SEED + epoch
        ).permutation(len(data))

        first_batch = (
            start_batch if epoch == start_epoch else 0
        )

        for batch_index in range(
            first_batch, batches_per_epoch
        ):
            begin = batch_index * batch_size
            end = min(begin + batch_size, len(data))

            positions = permutation[begin:end]

            encoded = tokenizer(
                [texts[int(i)] for i in positions],
                padding=True,
                truncation=True,
                max_length=settings["max_length"],
                return_tensors="pt"
            )

            encoded = {
                key: value.to(device)
                for key, value in encoded.items()
            }

            targets = torch.tensor(
                labels[positions],
                dtype=torch.long,
                device=device
            )

            outputs = model(**encoded, labels=targets)
            loss = outputs.loss

            assert torch.isfinite(loss).item()

            loss_sum += loss.item() * len(positions)
            examples_seen += len(positions)

            remaining_group = (
                batches_per_epoch
                - (batch_index // accumulation) * accumulation
            )

            group_size = min(
                accumulation,
                remaining_group
            )

            (loss / group_size).backward()

            del outputs, loss, encoded, targets

            update_due = (
                (batch_index + 1) % accumulation == 0
                or batch_index + 1 == batches_per_epoch
            )

            if update_due:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                if global_step % 100 == 0:
                    print(
                        f"Epoch {epoch + 1}/{epochs} | "
                        f"Optimizer updates: "
                        f"{global_step}/{total_updates}"
                    )

                    save_checkpoint(
                        epoch,
                        batch_index + 1
                    )

        epoch_loss = loss_sum / examples_seen
        epoch_losses.append(
            round(float(epoch_loss), 6)
        )

        print(
            f"EPOCH {epoch + 1} COMPLETED | "
            f"Mean training loss: {epoch_loss:.6f}"
        )

        loss_sum = 0.0
        examples_seen = 0

        save_checkpoint(epoch + 1, 0)

    assert global_step == total_updates
    assert len(epoch_losses) == epochs

    model.eval()
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        MODEL_DIR,
        safe_serialization=True
    )
    tokenizer.save_pretrained(MODEL_DIR)

    weights_path = MODEL_DIR / "model.safetensors"
    assert weights_path.is_file()

    no_decay = (
        "bias", "LayerNorm.weight", "layer_norm.weight"
    )

    summary = {
        "status": "TRAINING_COMPLETED",
        "condition": CONDITION,
        "seed": SEED,
        "records": len(data),
        "epochs": epochs,
        "optimizer_updates": global_step,
        "warmup_updates": warmup_steps,
        "epoch_losses": epoch_losses,
        "model_id": cfg["model_id"],
        "model_revision_sha": cfg["model_revision_sha"],
        "protocol_sha256": protocol_hash,
        "training_csv_sha256": data_hash,
        "model_sha256": sha256_file(weights_path),
        "precision": "FLOAT32",
        "gradient_clipping": "NONE",
        "batch_order": "NUMPY_DEFAULT_RNG_SEED_PLUS_EPOCH",
        "optimizer": "ADAMW_DECOUPLED_WEIGHT_DECAY",
        "weight_decay_exclusions": list(no_decay),
        "recovery_mode": recovery_mode,
        "evaluation_performed": False,
        "holdout_accessed": False,
        "reference_labels_accessed": False
    }

    temp_summary = SUMMARY_PATH.with_suffix(".json.tmp")

    temp_summary.write_text(
        json.dumps(summary, indent=2) + "\n",
        encoding="utf-8"
    )

    os.replace(temp_summary, SUMMARY_PATH)

    assert SUMMARY_PATH.is_file()

    # Delete the resumable file only after completion.
    CHECKPOINT_PATH.unlink(missing_ok=True)

    print("\nSTAGE 13 — CORRECTED SEED 42 COMPLETED")
    print("Recovery mode:", recovery_mode)
    print("Epoch losses:", epoch_losses)
    print("Optimizer updates:", global_step)
    print("Final model:", MODEL_DIR)
    print("Training reference labels accessed: NO")
    print("Holdout accessed: NO")
    print("Evaluation performed: NO")
    print("\nSTATUS: CORRECTED SEED 42 TRAINING PASSED")


run_corrected_training()


STAGE 13 — S-SLTE CORRECTED TRAINING
Condition: corrected
Seed: 42
Recovery mode: IN_MEMORY
Starting optimizer update: 700
Total optimizer updates: 852
Training records: 13621
Changed labels: 705
GPU: Tesla T4
Holdout accessed: NO
Epoch 2/2 | Optimizer updates: 800/852
EPOCH 2 COMPLETED | Mean training loss: 0.171706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — CORRECTED SEED 42 COMPLETED
Recovery mode: IN_MEMORY
Epoch losses: [0.394695, 0.171706]
Optimizer updates: 852
Final model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/corrected_seed_42/final_model
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: CORRECTED SEED 42 TRAINING PASSED


In [ ]:

import json
import hashlib
from pathlib import Path

from transformers import AutoConfig

# Stage 13 — Verify completed matched training pair.
BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    protocol = json.load(f)

def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

protocol_hash = file_sha256(PROTOCOL_PATH)

assert protocol["model_id"] == "distilbert/distilbert-base-uncased"
assert protocol["seeds"] == [42, 123, 2026]
assert protocol["training"]["epochs"] == 2

SEED = 42
summaries = {}

for condition in ("baseline", "corrected"):

    run_dir = (
        ROOT / "training_runs"
        / f"{condition}_seed_{SEED}"
    )

    summary_path = run_dir / "training_summary_v1.json"
    model_dir = run_dir / "final_model"
    weights_path = model_dir / "model.safetensors"
    config_path = model_dir / "config.json"

    assert summary_path.is_file()
    assert weights_path.is_file()
    assert config_path.is_file()

    with summary_path.open(encoding="utf-8") as f:
        summary = json.load(f)

    # Read the saved Hugging Face model configuration.
    model_config = AutoConfig.from_pretrained(
        str(model_dir),
        local_files_only=True
    )

    assert model_config.num_labels == 4
    assert model_config.model_type == "distilbert"

    assert summary["status"] == "TRAINING_COMPLETED"
    assert summary["condition"] == condition
    assert summary["seed"] == SEED
    assert summary["records"] == 13621
    assert summary["epochs"] == 2
    assert summary["optimizer_updates"] == 852
    assert summary["warmup_updates"] == 52

    assert summary["model_id"] == protocol["model_id"]
    assert summary["model_revision_sha"] == (
        protocol["model_revision_sha"]
    )

    assert summary["protocol_sha256"] == protocol_hash

    expected_data_hash = protocol["input_sha256"][
        f"{condition}_csv"
    ]

    assert summary["training_csv_sha256"] == (
        expected_data_hash
    )

    assert len(summary["model_sha256"]) == 64
    assert len(summary["epoch_losses"]) == 2
    assert all(
        isinstance(loss, (int, float)) and loss >= 0
        for loss in summary["epoch_losses"]
    )

    assert weights_path.stat().st_size > 100_000_000

    assert summary["evaluation_performed"] is False
    assert summary["holdout_accessed"] is False
    assert summary["reference_labels_accessed"] is False

    summaries[condition] = summary

baseline = summaries["baseline"]
corrected = summaries["corrected"]

# Verify matched experiment settings.
assert baseline["seed"] == corrected["seed"]
assert baseline["epochs"] == corrected["epochs"]
assert baseline["optimizer_updates"] == (
    corrected["optimizer_updates"]
)
assert baseline["protocol_sha256"] == (
    corrected["protocol_sha256"]
)
assert baseline["model_revision_sha"] == (
    corrected["model_revision_sha"]
)

assert baseline["epoch_losses"] == [
    0.451491, 0.230441
]

assert corrected["epoch_losses"] == [
    0.394695, 0.171706
]

print("STAGE 13 — MATCHED SEED 42 VERIFICATION")

print("\nBASELINE")
print("Epoch losses:", baseline["epoch_losses"])
print("Final model: FOUND")
print("Model configuration: VERIFIED")

print("\nS-SLTE CORRECTED")
print("Epoch losses:", corrected["epoch_losses"])
print("Final model: FOUND")
print("Model configuration: VERIFIED")
print(
    "Recovery mode:",
    corrected.get("recovery_mode", "NOT_RECORDED")
)

print("\nMATCHED TRAINING")
print("Same pretrained revision: YES")
print("Same random seed: YES")
print("Same frozen protocol: YES")
print("Same epochs: YES")
print("Same optimizer update count: YES")
print("Number of output classes: 4")
print("Model artifacts: FOUND")
print("Training summaries: VERIFIED")

print("\nRESEARCH INTEGRITY")
print("Training reference labels accessed: NO")
print("Holdout accessed: NO")
print("Evaluation performed: NO")
print("Model weights independently rehashed: NO")
print("Original artifacts modified: NO")

print("\nSTATUS: MATCHED SEED 42 ARTIFACTS VERIFIED")


STAGE 13 — MATCHED SEED 42 VERIFICATION

BASELINE
Epoch losses: [0.451491, 0.230441]
Final model: FOUND
Model configuration: VERIFIED

S-SLTE CORRECTED
Epoch losses: [0.394695, 0.171706]
Final model: FOUND
Model configuration: VERIFIED
Recovery mode: IN_MEMORY

MATCHED TRAINING
Same pretrained revision: YES
Same random seed: YES
Same frozen protocol: YES
Same epochs: YES
Same optimizer update count: YES
Number of output classes: 4
Model artifacts: FOUND
Training summaries: VERIFIED

RESEARCH INTEGRITY
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO
Model weights independently rehashed: NO
Original artifacts modified: NO

STATUS: MATCHED SEED 42 ARTIFACTS VERIFIED


In [ ]:

%pip install -q "transformers==5.20.1" "accelerate" "safetensors"

import os
import gc
import json
import math
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"
PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"

if not PROTOCOL_PATH.is_file():
    drive.mount("/content/drive")

assert PROTOCOL_PATH.is_file(), "Frozen protocol unavailable."
assert torch.cuda.is_available(), "Enable T4 GPU."
assert transformers.__version__ == "5.20.1"

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    TRAINING_PROTOCOL = json.load(f)

assert TRAINING_PROTOCOL["model_id"] == (
    "distilbert/distilbert-base-uncased"
)
assert TRAINING_PROTOCOL["model_revision_sha"] == (
    "12040accade4e8a0f71eabdb258fecc2e7e948be"
)
assert TRAINING_PROTOCOL["seeds"] == [42, 123, 2026]
assert TRAINING_PROTOCOL["training"]["epochs"] == 2

SETTINGS = TRAINING_PROTOCOL["training"]

assert SETTINGS["train_batch_size"] == 16
assert SETTINGS["gradient_accumulation_steps"] == 2
assert SETTINGS["learning_rate"] == 2e-5
assert SETTINGS["warmup_ratio"] == 0.06
assert SETTINGS["weight_decay"] == 0.01
assert SETTINGS["max_length"] == 256
assert SETTINGS["lr_scheduler"] == "linear"
assert SETTINGS["optimizer"] == "AdamW"
assert SETTINGS["early_stopping"] is False
assert SETTINGS["drop_last"] is False
assert SETTINGS["final_checkpoint_policy"] == "LAST_EPOCH"

def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

PROTOCOL_HASH = file_sha256(PROTOCOL_PATH)

def run_matched_training(seed, condition):

    assert seed in TRAINING_PROTOCOL["seeds"]
    assert seed in (123, 2026)
    assert condition in ("baseline", "corrected")

    data_path = ROOT / f"{condition}_training_v1.csv"
    pair_condition = (
        "corrected" if condition == "baseline" else "baseline"
    )
    paired_path = ROOT / f"{pair_condition}_training_v1.csv"

    run_dir = (
        ROOT / "training_runs"
        / f"{condition}_seed_{seed}"
    )

    checkpoint_path = run_dir / "training_checkpoint.pt"
    model_dir = run_dir / "final_model"
    summary_path = run_dir / "training_summary_v1.json"

    local_dir = Path(
        f"/content/s_slte_training_{condition}_{seed}"
    )

    local_dir.mkdir(parents=True, exist_ok=True)
    run_dir.mkdir(parents=True, exist_ok=True)

    expected_hash = TRAINING_PROTOCOL[
        "input_sha256"
    ][f"{condition}_csv"]

    expected_pair_hash = TRAINING_PROTOCOL[
        "input_sha256"
    ][f"{pair_condition}_csv"]

    assert file_sha256(PROTOCOL_PATH) == PROTOCOL_HASH

    # Verify completed runs without retraining.
    if summary_path.exists():
        with summary_path.open(encoding="utf-8") as f:
            completed = json.load(f)

        assert completed["status"] == "TRAINING_COMPLETED"
        assert completed["condition"] == condition
        assert completed["seed"] == seed
        assert completed["protocol_sha256"] == PROTOCOL_HASH
        assert completed["training_csv_sha256"] == expected_hash
        assert completed["optimizer_updates"] == 852
        assert len(completed["epoch_losses"]) == 2

        weights = model_dir / "model.safetensors"
        assert weights.is_file()
        assert (model_dir / "config.json").is_file()
        assert weights.stat().st_size > 100_000_000

        print("STAGE 13 — COMPLETED RUN VERIFIED")
        print("Condition:", condition)
        print("Seed:", seed)
        print("Epoch losses:", completed["epoch_losses"])
        print("No retraining performed.")
        return

    # Read Drive inputs once; train using local copies.
    local_data = local_dir / data_path.name
    local_pair = local_dir / paired_path.name

    if not local_data.exists():
        shutil.copyfile(data_path, local_data)

    if not local_pair.exists():
        shutil.copyfile(paired_path, local_pair)

    assert file_sha256(local_data) == expected_hash
    assert file_sha256(local_pair) == expected_pair_hash

    data = pd.read_csv(local_data)
    paired = pd.read_csv(local_pair)

    assert len(data) == len(paired) == 13621
    assert data["record_id"].is_unique
    assert data["record_id"].equals(paired["record_id"])
    assert data["text"].equals(paired["text"])
    assert data["text"].notna().all()
    assert data["label"].isin(range(4)).all()
    assert paired["label"].isin(range(4)).all()

    changed = int(
        (data["label"] != paired["label"]).sum()
    )
    assert changed == 705

    texts = data["text"].astype(str).tolist()
    labels = data["label"].to_numpy(dtype=np.int64)

    batch_size = SETTINGS["train_batch_size"]
    accumulation = SETTINGS["gradient_accumulation_steps"]
    epochs = SETTINGS["epochs"]

    batches_per_epoch = math.ceil(len(data) / batch_size)
    updates_per_epoch = math.ceil(
        batches_per_epoch / accumulation
    )

    total_updates = updates_per_epoch * epochs
    warmup_steps = math.ceil(
        total_updates * SETTINGS["warmup_ratio"]
    )

    assert batches_per_epoch == 852
    assert total_updates == 852
    assert warmup_steps == 52

    # Release previous GPU objects.
    for name in (
        "model", "downstream_model",
        "optimizer", "scheduler"
    ):
        if name in globals():
            del globals()[name]

    gc.collect()
    torch.cuda.empty_cache()

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

    device = torch.device("cuda:0")

    tokenizer = AutoTokenizer.from_pretrained(
        TRAINING_PROTOCOL["model_id"],
        revision=TRAINING_PROTOCOL["model_revision_sha"],
        token=False,
        use_fast=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        TRAINING_PROTOCOL["model_id"],
        revision=TRAINING_PROTOCOL["model_revision_sha"],
        token=False,
        num_labels=4
    ).to(device)

    no_decay = (
        "bias", "LayerNorm.weight", "layer_norm.weight"
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if p.requires_grad
                    and not any(key in n for key in no_decay)
                ],
                "weight_decay": SETTINGS["weight_decay"]
            },
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if p.requires_grad
                    and any(key in n for key in no_decay)
                ],
                "weight_decay": 0.0
            }
        ],
        lr=SETTINGS["learning_rate"],
        foreach=False
    )

    def lr_multiplier(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        return max(
            0.0,
            (total_updates - step)
            / max(1, total_updates - warmup_steps)
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_multiplier
    )

    start_epoch = 0
    start_batch = 0
    global_step = 0
    epoch_losses = []
    loss_sum = 0.0
    examples_seen = 0
    recovery_mode = "FRESH"

    # Resume only from a verified completed-update checkpoint.
    if checkpoint_path.exists():
        state = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        assert state["seed"] == seed
        assert state["condition"] == condition
        assert state["protocol_sha256"] == PROTOCOL_HASH
        assert state["training_csv_sha256"] == expected_hash

        start_epoch = int(state["epoch"])
        start_batch = int(state["next_batch"])
        global_step = int(state["global_step"])

        assert 0 <= start_epoch <= epochs
        assert 0 <= start_batch < batches_per_epoch
        assert global_step == (
            start_epoch * updates_per_epoch
            + math.ceil(start_batch / accumulation)
        )
        assert start_batch % accumulation == 0

        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])

        epoch_losses = list(state["epoch_losses"])
        loss_sum = float(state["loss_sum"])
        examples_seen = int(state["examples_seen"])

        assert len(epoch_losses) == start_epoch

        torch.set_rng_state(state["cpu_rng_state"])
        torch.cuda.set_rng_state_all(
            state["cuda_rng_states"]
        )

        recovery_mode = "DISK_CHECKPOINT"

        del state

    def save_checkpoint(next_epoch, next_batch):
        state = {
            "seed": seed,
            "condition": condition,
            "protocol_sha256": PROTOCOL_HASH,
            "training_csv_sha256": expected_hash,
            "epoch": next_epoch,
            "next_batch": next_batch,
            "global_step": global_step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "cpu_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all(),
            "epoch_losses": list(epoch_losses),
            "loss_sum": loss_sum,
            "examples_seen": examples_seen
        }

        local_tmp = local_dir / "checkpoint.pt"
        drive_tmp = checkpoint_path.with_suffix(".pt.tmp")

        torch.save(state, local_tmp)
        shutil.copyfile(local_tmp, drive_tmp)
        os.replace(drive_tmp, checkpoint_path)
        local_tmp.unlink(missing_ok=True)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    print("STAGE 13 — DOWNSTREAM TRAINING STARTED")
    print("Condition:", condition)
    print("Seed:", seed)
    print("Recovery mode:", recovery_mode)
    print("Training records:", len(data))
    print("Changed labels:", changed)
    print("Epochs:", epochs)
    print("Total optimizer updates:", total_updates)
    print("Warmup updates:", warmup_steps)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Holdout accessed: NO")

    for epoch in range(start_epoch, epochs):

        permutation = np.random.default_rng(
            seed + epoch
        ).permutation(len(data))

        first_batch = (
            start_batch if epoch == start_epoch else 0
        )

        for batch_index in range(
            first_batch, batches_per_epoch
        ):
            begin = batch_index * batch_size
            end = min(begin + batch_size, len(data))

            positions = permutation[begin:end]

            encoded = tokenizer(
                [texts[int(i)] for i in positions],
                padding=True,
                truncation=True,
                max_length=SETTINGS["max_length"],
                return_tensors="pt"
            )

            encoded = {
                key: value.to(device)
                for key, value in encoded.items()
            }

            targets = torch.tensor(
                labels[positions],
                dtype=torch.long,
                device=device
            )

            outputs = model(**encoded, labels=targets)
            loss = outputs.loss

            assert torch.isfinite(loss).item()

            loss_sum += loss.item() * len(positions)
            examples_seen += len(positions)

            remaining_group = (
                batches_per_epoch
                - (batch_index // accumulation) * accumulation
            )
            group_size = min(
                accumulation, remaining_group
            )

            (loss / group_size).backward()

            del outputs, loss, encoded, targets

            update_due = (
                (batch_index + 1) % accumulation == 0
                or batch_index + 1 == batches_per_epoch
            )

            if update_due:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                if global_step % 100 == 0:
                    print(
                        f"Epoch {epoch + 1}/{epochs} | "
                        f"Optimizer updates: "
                        f"{global_step}/{total_updates}"
                    )

                # Fewer large Drive writes; training is unchanged.
                if global_step % 200 == 0:
                    save_checkpoint(
                        epoch,
                        batch_index + 1
                    )

        epoch_loss = loss_sum / examples_seen
        epoch_losses.append(round(float(epoch_loss), 6))

        print(
            f"EPOCH {epoch + 1} COMPLETED | "
            f"Mean loss: {epoch_loss:.6f}"
        )

        loss_sum = 0.0
        examples_seen = 0

        if epoch + 1 < epochs:
            save_checkpoint(epoch + 1, 0)

    assert global_step == total_updates
    assert len(epoch_losses) == epochs

    # Save locally before transferring final files to Drive.
    model.eval()

    local_model_dir = local_dir / "final_model"
    local_model_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        local_model_dir,
        safe_serialization=True
    )
    tokenizer.save_pretrained(local_model_dir)

    local_weights = local_model_dir / "model.safetensors"
    assert local_weights.is_file()

    model_hash = file_sha256(local_weights)

    if not model_dir.exists():
        shutil.copytree(local_model_dir, model_dir)
    else:
        # An incomplete previous save can be repaired.
        assert not summary_path.exists()
        shutil.copytree(
            local_model_dir,
            model_dir,
            dirs_exist_ok=True
        )

    assert (model_dir / "model.safetensors").is_file()
    assert (
        (model_dir / "model.safetensors").stat().st_size
        == local_weights.stat().st_size
    )

    summary = {
        "status": "TRAINING_COMPLETED",
        "condition": condition,
        "seed": seed,
        "records": len(data),
        "epochs": epochs,
        "optimizer_updates": global_step,
        "warmup_updates": warmup_steps,
        "epoch_losses": epoch_losses,
        "model_id": TRAINING_PROTOCOL["model_id"],
        "model_revision_sha": TRAINING_PROTOCOL[
            "model_revision_sha"
        ],
        "protocol_sha256": PROTOCOL_HASH,
        "training_csv_sha256": expected_hash,
        "model_sha256": model_hash,
        "precision": "FLOAT32",
        "gradient_clipping": "NONE",
        "batch_order": "NUMPY_DEFAULT_RNG_SEED_PLUS_EPOCH",
        "optimizer": "ADAMW_DECOUPLED_WEIGHT_DECAY",
        "weight_decay_exclusions": list(no_decay),
        "recovery_mode": recovery_mode,
        "evaluation_performed": False,
        "holdout_accessed": False,
        "reference_labels_accessed": False
    }

    summary_tmp = summary_path.with_suffix(".json.tmp")
    summary_tmp.write_text(
        json.dumps(summary, indent=2) + "\n",
        encoding="utf-8"
    )
    os.replace(summary_tmp, summary_path)

    assert summary_path.exists()

    checkpoint_path.unlink(missing_ok=True)

    print("\nSTAGE 13 — TRAINING COMPLETED")
    print("Condition:", condition)
    print("Seed:", seed)
    print("Epoch losses:", epoch_losses)
    print("Optimizer updates:", global_step)
    print("Recovery mode:", recovery_mode)
    print("Final model:", model_dir)
    print("Summary:", summary_path)
    print("Training reference labels accessed: NO")
    print("Holdout accessed: NO")
    print("Evaluation performed: NO")
    print("\nSTATUS: DOWNSTREAM TRAINING PASSED")

# Run 3 of 6: Original LLM-label baseline, seed 123.
run_matched_training(seed=123, condition="baseline")


Mounted at /content/drive


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

... (2 outputs left out)

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — DOWNSTREAM TRAINING STARTED
Condition: baseline
Seed: 123
Recovery mode: FRESH
Training records: 13621
Changed labels: 705
Epochs: 2
Total optimizer updates: 852
Warmup updates: 52
GPU: Tesla T4
Holdout accessed: NO
Epoch 1/2 | Optimizer updates: 100/852
Epoch 1/2 | Optimizer updates: 200/852
Epoch 1/2 | Optimizer updates: 300/852
Epoch 1/2 | Optimizer updates: 400/852
EPOCH 1 COMPLETED | Mean loss: 0.458791
Epoch 2/2 | Optimizer updates: 500/852
Epoch 2/2 | Optimizer updates: 600/852
Epoch 2/2 | Optimizer updates: 700/852
Epoch 2/2 | Optimizer updates: 800/852
EPOCH 2 COMPLETED | Mean loss: 0.228704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — TRAINING COMPLETED
Condition: baseline
Seed: 123
Epoch losses: [0.458791, 0.228704]
Optimizer updates: 852
Recovery mode: FRESH
Final model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_123/final_model
Summary: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_123/training_summary_v1.json
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: DOWNSTREAM TRAINING PASSED


In [ ]:

# Stage 13 — Run 4 of 6: S-SLTE corrected labels, seed 123.
run_matched_training(seed=123, condition="corrected")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — DOWNSTREAM TRAINING STARTED
Condition: corrected
Seed: 123
Recovery mode: FRESH
Training records: 13621
Changed labels: 705
Epochs: 2
Total optimizer updates: 852
Warmup updates: 52
GPU: Tesla T4
Holdout accessed: NO
Epoch 1/2 | Optimizer updates: 100/852
Epoch 1/2 | Optimizer updates: 200/852
Epoch 1/2 | Optimizer updates: 300/852
Epoch 1/2 | Optimizer updates: 400/852
EPOCH 1 COMPLETED | Mean loss: 0.404000
Epoch 2/2 | Optimizer updates: 500/852
Epoch 2/2 | Optimizer updates: 600/852
Epoch 2/2 | Optimizer updates: 700/852
Epoch 2/2 | Optimizer updates: 800/852
EPOCH 2 COMPLETED | Mean loss: 0.168965


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — TRAINING COMPLETED
Condition: corrected
Seed: 123
Epoch losses: [0.404, 0.168965]
Optimizer updates: 852
Recovery mode: FRESH
Final model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/corrected_seed_123/final_model
Summary: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/corrected_seed_123/training_summary_v1.json
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: DOWNSTREAM TRAINING PASSED


In [ ]:

import json
import hashlib
from pathlib import Path
from transformers import AutoConfig

# Stage 13 — Verify matched training pair for seed 123.
ROOT = Path(
    "/content/drive/MyDrive/S_SLTE_Thesis/"
    "outputs/s_slte_downstream_v1"
)

protocol_path = ROOT / "downstream_training_protocol_v1.json"

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

with protocol_path.open(encoding="utf-8") as f:
    protocol = json.load(f)

protocol_hash = sha256_file(protocol_path)

seed = 123
summaries = {}

for condition in ["baseline", "corrected"]:
    run_dir = (
        ROOT / "training_runs"
        / f"{condition}_seed_{seed}"
    )

    summary_path = run_dir / "training_summary_v1.json"
    model_dir = run_dir / "final_model"
    weights_path = model_dir / "model.safetensors"

    assert summary_path.is_file()
    assert weights_path.is_file()

    with summary_path.open(encoding="utf-8") as f:
        summary = json.load(f)

    model_config = AutoConfig.from_pretrained(
        str(model_dir),
        local_files_only=True
    )

    assert model_config.num_labels == 4
    assert model_config.model_type == "distilbert"

    assert summary["status"] == "TRAINING_COMPLETED"
    assert summary["condition"] == condition
    assert summary["seed"] == seed

    assert summary["records"] == 13621
    assert summary["epochs"] == 2
    assert summary["optimizer_updates"] == 852
    assert summary["warmup_updates"] == 52

    assert summary["protocol_sha256"] == protocol_hash
    assert summary["model_id"] == protocol["model_id"]
    assert summary["model_revision_sha"] == (
        protocol["model_revision_sha"]
    )

    assert summary["training_csv_sha256"] == (
        protocol["input_sha256"][f"{condition}_csv"]
    )

    assert len(summary["epoch_losses"]) == 2
    assert len(summary["model_sha256"]) == 64
    assert weights_path.stat().st_size > 100_000_000

    assert summary["holdout_accessed"] is False
    assert summary["evaluation_performed"] is False
    assert summary["reference_labels_accessed"] is False

    summaries[condition] = summary

baseline = summaries["baseline"]
corrected = summaries["corrected"]

assert baseline["protocol_sha256"] == (
    corrected["protocol_sha256"]
)
assert baseline["seed"] == corrected["seed"]
assert baseline["optimizer_updates"] == (
    corrected["optimizer_updates"]
)
assert baseline["model_revision_sha"] == (
    corrected["model_revision_sha"]
)

assert baseline["epoch_losses"] == [
    0.458791, 0.228704
]
assert corrected["epoch_losses"] == [
    0.404, 0.168965
]

print("STAGE 13 — MATCHED SEED 123 VERIFICATION")

print("\nBASELINE")
print("Epoch losses:", baseline["epoch_losses"])
print("Model artifacts: FOUND")

print("\nS-SLTE CORRECTED")
print("Epoch losses:", corrected["epoch_losses"])
print("Model artifacts: FOUND")

print("\nMATCHED TRAINING")
print("Same random seed: YES")
print("Same frozen protocol: YES")
print("Same pretrained model revision: YES")
print("Same training records: 13621")
print("Same optimizer update count: 852")

print("\nRESEARCH INTEGRITY")
print("Holdout accessed: NO")
print("Evaluation performed: NO")
print("Reference labels accessed: NO")
print("Model weights independently rehashed: NO")

print("\nSTATUS: MATCHED SEED 123 ARTIFACTS VERIFIED")


STAGE 13 — MATCHED SEED 123 VERIFICATION

BASELINE
Epoch losses: [0.458791, 0.228704]
Model artifacts: FOUND

S-SLTE CORRECTED
Epoch losses: [0.404, 0.168965]
Model artifacts: FOUND

MATCHED TRAINING
Same random seed: YES
Same frozen protocol: YES
Same pretrained model revision: YES
Same training records: 13621
Same optimizer update count: 852

RESEARCH INTEGRITY
Holdout accessed: NO
Evaluation performed: NO
Reference labels accessed: NO
Model weights independently rehashed: NO

STATUS: MATCHED SEED 123 ARTIFACTS VERIFIED


In [ ]:

# Stage 13 — Run 5 of 6
# Baseline DistilBERT training with random seed 2026.

run_matched_training(seed=2026, condition="baseline")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — DOWNSTREAM TRAINING STARTED
Condition: baseline
Seed: 2026
Recovery mode: FRESH
Training records: 13621
Changed labels: 705
Epochs: 2
Total optimizer updates: 852
Warmup updates: 52
GPU: Tesla T4
Holdout accessed: NO
Epoch 1/2 | Optimizer updates: 100/852
Epoch 1/2 | Optimizer updates: 200/852
Epoch 1/2 | Optimizer updates: 300/852
Epoch 1/2 | Optimizer updates: 400/852
EPOCH 1 COMPLETED | Mean loss: 0.458585
Epoch 2/2 | Optimizer updates: 500/852
Epoch 2/2 | Optimizer updates: 600/852
Epoch 2/2 | Optimizer updates: 700/852
Epoch 2/2 | Optimizer updates: 800/852
EPOCH 2 COMPLETED | Mean loss: 0.224221


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — TRAINING COMPLETED
Condition: baseline
Seed: 2026
Epoch losses: [0.458585, 0.224221]
Optimizer updates: 852
Recovery mode: FRESH
Final model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_2026/final_model
Summary: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_2026/training_summary_v1.json
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: DOWNSTREAM TRAINING PASSED


In [ ]:

# Stage 13 — Run 6 of 6
# Final matched training run: S-SLTE corrected labels, seed 2026.

run_matched_training(seed=2026, condition="corrected")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — DOWNSTREAM TRAINING STARTED
Condition: corrected
Seed: 2026
Recovery mode: FRESH
Training records: 13621
Changed labels: 705
Epochs: 2
Total optimizer updates: 852
Warmup updates: 52
GPU: Tesla T4
Holdout accessed: NO
Epoch 1/2 | Optimizer updates: 100/852
Epoch 1/2 | Optimizer updates: 200/852
Epoch 1/2 | Optimizer updates: 300/852
Epoch 1/2 | Optimizer updates: 400/852
EPOCH 1 COMPLETED | Mean loss: 0.402684
Epoch 2/2 | Optimizer updates: 500/852
Epoch 2/2 | Optimizer updates: 600/852
Epoch 2/2 | Optimizer updates: 700/852
Epoch 2/2 | Optimizer updates: 800/852
EPOCH 2 COMPLETED | Mean loss: 0.170546


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — TRAINING COMPLETED
Condition: corrected
Seed: 2026
Epoch losses: [0.402684, 0.170546]
Optimizer updates: 852
Recovery mode: FRESH
Final model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/corrected_seed_2026/final_model
Summary: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/corrected_seed_2026/training_summary_v1.json
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: DOWNSTREAM TRAINING PASSED


In [1]:

# Final integrity audit of all six downstream training runs.
# No training, prediction, or holdout access.

import os
import json
import hashlib
from pathlib import Path

import pandas as pd
from transformers import AutoConfig
from safetensors import safe_open
from google.colab import drive

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"
PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"

if not PROTOCOL_PATH.is_file():
    drive.mount("/content/drive")

assert PROTOCOL_PATH.is_file(), "Frozen protocol not found."

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    protocol = json.load(f)

protocol_hash = sha256_file(PROTOCOL_PATH)

SEEDS = [42, 123, 2026]
CONDITIONS = ["baseline", "corrected"]

assert protocol["seeds"] == SEEDS
assert protocol["training"]["epochs"] == 2
assert protocol["model_id"] == (
    "distilbert/distilbert-base-uncased"
)
assert protocol["model_revision_sha"] == (
    "12040accade4e8a0f71eabdb258fecc2e7e948be"
)

# Independently verify both frozen training CSVs.
input_hashes = {}

for condition in CONDITIONS:
    csv_path = ROOT / f"{condition}_training_v1.csv"
    assert csv_path.is_file()

    actual_hash = sha256_file(csv_path)
    expected_hash = protocol["input_sha256"][
        f"{condition}_csv"
    ]

    assert actual_hash == expected_hash

    input_hashes[condition] = actual_hash

# Check that both datasets represent the same records.
baseline_data = pd.read_csv(
    ROOT / "baseline_training_v1.csv"
)
corrected_data = pd.read_csv(
    ROOT / "corrected_training_v1.csv"
)

assert len(baseline_data) == 13621
assert len(corrected_data) == 13621

assert baseline_data["record_id"].equals(
    corrected_data["record_id"]
)

assert baseline_data["text"].equals(
    corrected_data["text"]
)

changed_labels = int(
    (
        baseline_data["label"]
        != corrected_data["label"]
    ).sum()
)

assert changed_labels == 705

rows = []
run_summaries = {}

for seed in SEEDS:
    for condition in CONDITIONS:

        run_dir = (
            ROOT / "training_runs"
            / f"{condition}_seed_{seed}"
        )

        summary_path = (
            run_dir / "training_summary_v1.json"
        )

        model_dir = run_dir / "final_model"
        weights_path = model_dir / "model.safetensors"

        assert summary_path.is_file(), (
            f"Missing summary: {condition}, seed {seed}"
        )

        assert weights_path.is_file(), (
            f"Missing weights: {condition}, seed {seed}"
        )

        assert (model_dir / "config.json").is_file()

        with summary_path.open(
            encoding="utf-8"
        ) as f:
            summary = json.load(f)

        config = AutoConfig.from_pretrained(
            str(model_dir),
            local_files_only=True
        )

        assert config.num_labels == 4
        assert config.model_type == "distilbert"

        assert summary["status"] == "TRAINING_COMPLETED"
        assert summary["seed"] == seed
        assert summary["condition"] == condition

        assert summary["records"] == 13621
        assert summary["epochs"] == 2
        assert summary["optimizer_updates"] == 852
        assert summary["warmup_updates"] == 52

        assert summary["model_id"] == protocol["model_id"]
        assert summary["model_revision_sha"] == (
            protocol["model_revision_sha"]
        )

        assert summary["protocol_sha256"] == protocol_hash
        assert summary["training_csv_sha256"] == (
            input_hashes[condition]
        )

        assert summary["holdout_accessed"] is False
        assert summary["evaluation_performed"] is False
        assert summary["reference_labels_accessed"] is False

        assert len(summary["epoch_losses"]) == 2
        assert len(summary["model_sha256"]) == 64

        assert weights_path.stat().st_size > 100_000_000

        # Inspect tensor structure without loading the full model.
        with safe_open(
            str(weights_path),
            framework="pt",
            device="cpu"
        ) as weights:

            keys = set(weights.keys())

            assert "classifier.weight" in keys
            assert "classifier.bias" in keys
            assert (
                "distilbert.embeddings.word_embeddings.weight"
                in keys
            )

            classifier_shape = weights.get_slice(
                "classifier.weight"
            ).get_shape()

            assert classifier_shape == [4, 768]

        run_summaries[(seed, condition)] = summary

        rows.append({
            "Seed": seed,
            "Condition": condition,
            "Epoch 1 Loss": summary["epoch_losses"][0],
            "Epoch 2 Loss": summary["epoch_losses"][1],
            "Updates": summary["optimizer_updates"],
            "Status": "VERIFIED"
        })

# Ensure distinct saved model fingerprints within each pair.
for seed in SEEDS:

    baseline = run_summaries[(seed, "baseline")]
    corrected = run_summaries[(seed, "corrected")]

    assert baseline["protocol_sha256"] == (
        corrected["protocol_sha256"]
    )

    assert baseline["model_revision_sha"] == (
        corrected["model_revision_sha"]
    )

    assert baseline["model_sha256"] != (
        corrected["model_sha256"]
    )

assert len(rows) == 6

results = pd.DataFrame(rows)

# Store a compact, machine-readable final training audit.
audit = {
    "stage": 13,
    "audit_version": 1,
    "status": "ALL_SIX_TRAINING_RUNS_AUDITED",
    "protocol_sha256": protocol_hash,
    "seeds": SEEDS,
    "conditions": CONDITIONS,
    "completed_runs": 6,
    "planned_runs": 6,
    "records_per_run": 13621,
    "changed_training_labels": 705,
    "optimizer_updates_per_run": 852,
    "input_sha256": input_hashes,
    "runs": rows,
    "model_structure_inspected": True,
    "saved_model_weights_independently_rehashed": False,
    "holdout_accessed": False,
    "holdout_evaluation_performed": False,
    "reference_labels_accessed": False
}

audit_path = ROOT / "final_training_audit_v1.json"
temporary_path = ROOT / "final_training_audit_v1.json.tmp"

temporary_path.write_text(
    json.dumps(audit, indent=2) + "\n",
    encoding="utf-8"
)

os.replace(temporary_path, audit_path)

print("STAGE 13 — FINAL TRAINING INTEGRITY AUDIT")
print()
print(results.to_string(index=False))

print("\nFROZEN EXPERIMENT")
print("Random seeds:", SEEDS)
print("Training records per run: 13621")
print("Corrected training labels: 705")
print("Protocol fingerprint: VERIFIED")
print("Training CSV fingerprints: VERIFIED")
print("Saved model structures: VERIFIED")
print("Completed training runs: 6/6")

print("\nRESEARCH INTEGRITY")
print("Holdout accessed: NO")
print("Holdout evaluation performed: NO")
print("Reference labels accessed: NO")
print("Saved weight contents independently hashed: NO")

print("\nAudit report:", audit_path)
print("\nSTATUS: ALL SIX TRAINING RUNS VERIFIED")


Mounted at /content/drive
STAGE 13 — FINAL TRAINING INTEGRITY AUDIT

 Seed Condition  Epoch 1 Loss  Epoch 2 Loss  Updates   Status
   42  baseline      0.451491      0.230441      852 VERIFIED
   42 corrected      0.394695      0.171706      852 VERIFIED
  123  baseline      0.458791      0.228704      852 VERIFIED
  123 corrected      0.404000      0.168965      852 VERIFIED
 2026  baseline      0.458585      0.224221      852 VERIFIED
 2026 corrected      0.402684      0.170546      852 VERIFIED

FROZEN EXPERIMENT
Random seeds: [42, 123, 2026]
Training records per run: 13621
Corrected training labels: 705
Protocol fingerprint: VERIFIED
Training CSV fingerprints: VERIFIED
Saved model structures: VERIFIED
Completed training runs: 6/6

RESEARCH INTEGRITY
Holdout accessed: NO
Holdout evaluation performed: NO
Reference labels accessed: NO
Saved weight contents independently hashed: NO

Audit report: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/final_training_audit_v1.

In [2]:

# Prepare the frozen internal holdout for evaluation.
# Read-only: no training, predictions, or file modifications.

import json
import hashlib
from pathlib import Path

import pandas as pd
from google.colab import drive

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"
AUDIT_PATH = ROOT / "final_training_audit_v1.json"
PARTITION_PATH = BASE / "configs/holdout_partition_manifest_v1.json"

SOURCE_PATH = BASE / "data/splits/noisy_training.csv"
REFERENCE_PATH = BASE / "private_reference/noisy_reference.csv"

if not PROTOCOL_PATH.is_file():
    drive.mount("/content/drive")

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

def fingerprint_ids(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def fingerprint_text(frame):
    h = hashlib.sha256()
    frame = frame.sort_values("record_id")

    for row in frame.itertuples(index=False):
        h.update(str(int(row.record_id)).encode("utf-8"))
        h.update(b"\0")
        h.update(str(row.text).encode("utf-8"))
        h.update(b"\0")

    return h.hexdigest()

# Gate: verify all six training runs before opening references.
with PROTOCOL_PATH.open(encoding="utf-8") as f:
    EVAL_PROTOCOL = json.load(f)

with AUDIT_PATH.open(encoding="utf-8") as f:
    EVAL_AUDIT = json.load(f)

EVAL_PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

assert EVAL_AUDIT["status"] == (
    "ALL_SIX_TRAINING_RUNS_AUDITED"
)
assert EVAL_AUDIT["completed_runs"] == 6
assert EVAL_AUDIT["planned_runs"] == 6
assert EVAL_AUDIT["protocol_sha256"] == EVAL_PROTOCOL_SHA

plan = EVAL_PROTOCOL["evaluation_plan"]

assert plan["primary_metric"] == "holdout_macro_f1"
assert plan["holdout_evaluation_timing"] == (
    "AFTER_ALL_SIX_TRAINING_RUNS"
)
assert plan["holdout_used_for_tuning"] is False

for seed in (42, 123, 2026):
    for condition in ("baseline", "corrected"):
        run_dir = (
            ROOT / "training_runs"
            / f"{condition}_seed_{seed}"
        )

        with (
            run_dir / "training_summary_v1.json"
        ).open(encoding="utf-8") as f:
            summary = json.load(f)

        assert summary["status"] == "TRAINING_COMPLETED"
        assert summary["seed"] == seed
        assert summary["condition"] == condition
        assert summary["optimizer_updates"] == 852
        assert summary["protocol_sha256"] == EVAL_PROTOCOL_SHA
        assert summary["holdout_accessed"] is False
        assert summary["evaluation_performed"] is False

        assert (
            run_dir / "final_model/model.safetensors"
        ).is_file()

# Load the partition manifest after training checks.
with PARTITION_PATH.open(encoding="utf-8") as f:
    EVAL_PARTITIONS = json.load(f)

spec = EVAL_PARTITIONS["partitions"]["holdout"]
holdout_ids = set(map(int, spec["record_ids"]))
training_ids = set(map(
    int,
    EVAL_PARTITIONS["partitions"]["training"]["record_ids"]
))

assert len(holdout_ids) == spec["count"] == 2000
assert fingerprint_ids(holdout_ids) == spec["id_sha256"]
assert holdout_ids.isdisjoint(training_ids)

for name, partition in EVAL_PARTITIONS["partitions"].items():
    if name == "holdout" or "record_ids" not in partition:
        continue

    other_ids = set(map(int, partition["record_ids"]))
    assert holdout_ids.isdisjoint(other_ids), (
        f"Holdout overlap with partition: {name}"
    )

# Restore the exact original holdout text.
source = pd.read_csv(
    SOURCE_PATH,
    usecols=["record_id", "text"]
)

assert len(source) == 16200
assert source["record_id"].is_unique
assert source["text"].notna().all()

holdout_text = (
    source[source["record_id"].isin(holdout_ids)]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(holdout_text) == 2000
assert set(map(int, holdout_text["record_id"])) == holdout_ids
assert fingerprint_text(holdout_text) == (
    spec["raw_text_sha256"]
)

# Access trusted labels ONLY for the frozen evaluation.
assert REFERENCE_PATH.is_file(), (
    "Original private reference file is missing."
)

reference = pd.read_csv(
    REFERENCE_PATH,
    usecols=["record_id", "text", "label"]
)

assert len(reference) == 16200
assert reference["record_id"].is_unique
assert reference["text"].notna().all()
assert reference["label"].notna().all()
assert reference["label"].isin([0, 1, 2, 3]).all()

reference_holdout = reference.loc[
    reference["record_id"].isin(holdout_ids),
    ["record_id", "text", "label"]
]

assert len(reference_holdout) == 2000

# Require identical texts in source and reference.
checked = holdout_text.merge(
    reference_holdout,
    on="record_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_source", "_reference")
)

assert len(checked) == 2000
assert (
    checked["text_source"]
    == checked["text_reference"]
).all()

# Retain only the evaluated IDs, original texts, and labels.
EVAL_HOLDOUT = (
    checked[
        ["record_id", "text_source", "label"]
    ]
    .rename(columns={"text_source": "text"})
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(EVAL_HOLDOUT) == 2000
assert EVAL_HOLDOUT["label"].isin([0, 1, 2, 3]).all()
assert fingerprint_text(
    EVAL_HOLDOUT[["record_id", "text"]]
) == spec["raw_text_sha256"]

print("STAGE 13 — FROZEN HOLDOUT PREPARATION")

print("\nTRAINING GATE")
print("Completed and audited training runs: 6/6")
print("Frozen training protocol: VERIFIED")
print("Training model weights modified: NO")

print("\nHOLDOUT INTEGRITY")
print("Frozen holdout records:", len(EVAL_HOLDOUT))
print("Record ID fingerprint: VERIFIED")
print("Original text fingerprint: VERIFIED")
print("Reference text alignment: VERIFIED")
print("Reference class labels: VERIFIED")
print("Training / holdout ID overlap: NONE")

print("\nEVALUATION PROTOCOL")
print("Primary metric:", plan["primary_metric"])
print("Evaluation timing: VERIFIED")
print("Holdout used for tuning: NO")

print("\nRESEARCH INTEGRITY")
print("Holdout reference labels accessed: YES — EVALUATION ONLY")
print("Model predictions generated: NO")
print("Holdout metrics calculated: NO")
print("Reference labels written to GitHub: NO")
print("Existing research files modified: NO")

print("\nSTATUS: FROZEN HOLDOUT READY FOR EVALUATION")


STAGE 13 — FROZEN HOLDOUT PREPARATION

TRAINING GATE
Completed and audited training runs: 6/6
Frozen training protocol: VERIFIED
Training model weights modified: NO

HOLDOUT INTEGRITY
Frozen holdout records: 2000
Record ID fingerprint: VERIFIED
Original text fingerprint: VERIFIED
Reference text alignment: VERIFIED
Reference class labels: VERIFIED
Training / holdout ID overlap: NONE

EVALUATION PROTOCOL
Primary metric: holdout_macro_f1
Evaluation timing: VERIFIED
Holdout used for tuning: NO

RESEARCH INTEGRITY
Holdout reference labels accessed: YES — EVALUATION ONLY
Model predictions generated: NO
Holdout metrics calculated: NO
Reference labels written to GitHub: NO
Existing research files modified: NO

STATUS: FROZEN HOLDOUT READY FOR EVALUATION


In [3]:

# STAGE 13 — STEP 15B
# Frozen holdout evaluation of all six DistilBERT models.
# No training, tuning, or correction-rule changes.

import os
import gc
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

EVAL_DIR = ROOT / "holdout_evaluation_v1"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

assert "EVAL_HOLDOUT" in globals(), (
    "Run Step 15A first to restore the frozen holdout."
)

assert torch.cuda.is_available(), "Enable T4 GPU."

# Load frozen protocol and six-run audit.
with (
    ROOT / "downstream_training_protocol_v1.json"
).open(encoding="utf-8") as f:
    protocol = json.load(f)

with (
    ROOT / "final_training_audit_v1.json"
).open(encoding="utf-8") as f:
    training_audit = json.load(f)

with (
    BASE / "configs/holdout_partition_manifest_v1.json"
).open(encoding="utf-8") as f:
    partition_manifest = json.load(f)

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

def id_fingerprint(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(
        index=False
    ):
        digest.update(str(int(row.record_id)).encode())
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

protocol_hash = sha256_file(
    ROOT / "downstream_training_protocol_v1.json"
)

assert training_audit["completed_runs"] == 6
assert training_audit["status"] == (
    "ALL_SIX_TRAINING_RUNS_AUDITED"
)
assert training_audit["protocol_sha256"] == protocol_hash

assert protocol["seeds"] == [42, 123, 2026]
assert protocol["num_labels"] == 4

plan = protocol["evaluation_plan"]

assert plan["primary_metric"] == "holdout_macro_f1"
assert plan["holdout_evaluation_timing"] == (
    "AFTER_ALL_SIX_TRAINING_RUNS"
)
assert plan["holdout_used_for_tuning"] is False

# Revalidate the in-memory frozen evaluation dataset.
holdout = EVAL_HOLDOUT.copy()

assert len(holdout) == 2000
assert holdout["record_id"].is_unique
assert holdout["text"].notna().all()
assert holdout["label"].isin([0, 1, 2, 3]).all()

holdout = (
    holdout.sort_values("record_id")
    .reset_index(drop=True)
)

holdout_spec = partition_manifest["partitions"]["holdout"]

record_ids = holdout["record_id"].to_numpy(dtype=np.int64)
y_true = holdout["label"].to_numpy(dtype=np.int64)

assert id_fingerprint(record_ids) == (
    holdout_spec["id_sha256"]
)

assert text_fingerprint(
    holdout[["record_id", "text"]]
) == holdout_spec["raw_text_sha256"]

training_ids = set(
    map(
        int,
        partition_manifest["partitions"]["training"]["record_ids"]
    )
)

assert set(map(int, record_ids)).isdisjoint(training_ids)

texts = holdout["text"].astype(str).tolist()
class_names = protocol["class_names"]

assert len(class_names) == 4

SEEDS = [42, 123, 2026]
CONDITIONS = ["baseline", "corrected"]
DEVICE = torch.device("cuda:0")

# Read the original final-model metadata, without modifying it.
run_info = {}

for seed in SEEDS:
    for condition in CONDITIONS:

        run_dir = (
            ROOT / "training_runs"
            / f"{condition}_seed_{seed}"
        )

        with (
            run_dir / "training_summary_v1.json"
        ).open(encoding="utf-8") as f:
            summary = json.load(f)

        assert summary["status"] == "TRAINING_COMPLETED"
        assert summary["seed"] == seed
        assert summary["condition"] == condition
        assert summary["optimizer_updates"] == 852
        assert summary["protocol_sha256"] == protocol_hash
        assert summary["holdout_accessed"] is False
        assert summary["evaluation_performed"] is False

        model_dir = run_dir / "final_model"

        assert (model_dir / "model.safetensors").is_file()
        assert (model_dir / "config.json").is_file()

        run_info[(seed, condition)] = {
            "model_dir": model_dir,
            "summary": summary
        }

print("STAGE 13 — FROZEN HOLDOUT EVALUATION")
print("Training runs verified: 6/6")
print("Holdout records:", len(holdout))
print("Primary metric: Macro-F1")
print("GPU:", torch.cuda.get_device_name(0))
print("Model training: DISABLED")
print("Hyperparameter tuning: DISABLED")
print()

# Each file contains record IDs and predictions only.
# No original reference labels are saved here.
all_predictions = pd.DataFrame({
    "record_id": record_ids
})

for seed in SEEDS:
    for condition in CONDITIONS:

        run_key = f"{condition}_seed_{seed}"

        model_dir = run_info[
            (seed, condition)
        ]["model_dir"]

        prediction_path = (
            EVAL_DIR / f"predictions_{run_key}.csv"
        )

        metadata_path = (
            EVAL_DIR / f"predictions_{run_key}_manifest.json"
        )

        column = f"pred_{condition}_seed_{seed}"

        if prediction_path.exists() or metadata_path.exists():

            assert prediction_path.is_file()
            assert metadata_path.is_file()

            with metadata_path.open(encoding="utf-8") as f:
                metadata = json.load(f)

            assert metadata["seed"] == seed
            assert metadata["condition"] == condition
            assert metadata["protocol_sha256"] == protocol_hash
            assert metadata["holdout_id_sha256"] == (
                holdout_spec["id_sha256"]
            )

            assert metadata["model_sha256_recorded"] == (
                run_info[(seed, condition)]["summary"][
                    "model_sha256"
                ]
            )

            assert sha256_file(prediction_path) == (
                metadata["prediction_file_sha256"]
            )

            saved = pd.read_csv(prediction_path)

            assert list(saved.columns) == [
                "record_id", "prediction"
            ]
            assert len(saved) == 2000
            assert np.array_equal(
                saved["record_id"].to_numpy(dtype=np.int64),
                record_ids
            )
            assert saved["prediction"].isin(range(4)).all()

            predictions = saved["prediction"].to_numpy(
                dtype=np.int64
            )

            print(f"{run_key}: EXISTING PREDICTIONS VERIFIED")

        else:
            gc.collect()
            torch.cuda.empty_cache()

            tokenizer = AutoTokenizer.from_pretrained(
                str(model_dir),
                local_files_only=True,
                use_fast=True
            )

            model = (
                AutoModelForSequenceClassification
                .from_pretrained(
                    str(model_dir),
                    local_files_only=True
                )
                .to(DEVICE)
            )

            assert model.config.num_labels == 4

            model.eval()
            prediction_batches = []

            # Inference batch size does not change training protocol.
            inference_batch_size = 32

            with torch.inference_mode():

                for start in range(
                    0,
                    len(texts),
                    inference_batch_size
                ):
                    batch_texts = texts[
                        start:start + inference_batch_size
                    ]

                    inputs = tokenizer(
                        batch_texts,
                        padding=True,
                        truncation=True,
                        max_length=256,
                        return_tensors="pt"
                    )

                    inputs = {
                        key: value.to(DEVICE)
                        for key, value in inputs.items()
                    }

                    logits = model(**inputs).logits

                    assert logits.shape == (
                        len(batch_texts), 4
                    )
                    assert torch.isfinite(logits).all().item()

                    batch_predictions = (
                        logits.argmax(dim=-1)
                        .cpu()
                        .numpy()
                    )

                    prediction_batches.append(batch_predictions)

                    del inputs, logits

            predictions = np.concatenate(
                prediction_batches
            ).astype(np.int64)

            assert len(predictions) == 2000
            assert np.isin(predictions, range(4)).all()

            output = pd.DataFrame({
                "record_id": record_ids,
                "prediction": predictions
            })

            temporary_prediction = prediction_path.with_suffix(
                ".csv.tmp"
            )

            temporary_prediction.write_text(
                output.to_csv(
                    index=False,
                    lineterminator="\n"
                ),
                encoding="utf-8"
            )

            os.replace(
                temporary_prediction,
                prediction_path
            )

            metadata = {
                "version": 1,
                "condition": condition,
                "seed": seed,
                "records": 2000,
                "protocol_sha256": protocol_hash,
                "holdout_id_sha256": holdout_spec["id_sha256"],
                "model_sha256_recorded": (
                    run_info[(seed, condition)]["summary"][
                        "model_sha256"
                    ]
                ),
                "prediction_file_sha256": sha256_file(
                    prediction_path
                ),
                "reference_labels_saved": False,
                "model_training_performed": False
            }

            metadata_temp = metadata_path.with_suffix(
                ".json.tmp"
            )

            metadata_temp.write_text(
                json.dumps(metadata, indent=2) + "\n",
                encoding="utf-8"
            )

            os.replace(metadata_temp, metadata_path)

            del model, tokenizer
            gc.collect()
            torch.cuda.empty_cache()

            print(f"{run_key}: PREDICTIONS SAVED")

        all_predictions[column] = predictions

assert len(all_predictions) == 2000
assert all_predictions.shape[1] == 7

print("\nALL SIX PREDICTION SETS READY")
print("Calculating frozen evaluation metrics...")

# Only now compare predictions with trusted reference labels.
metric_rows = []
per_class_rows = []

for seed in SEEDS:
    for condition in CONDITIONS:

        predictions = all_predictions[
            f"pred_{condition}_seed_{seed}"
        ].to_numpy(dtype=np.int64)

        accuracy = accuracy_score(
            y_true, predictions
        )

        macro_f1 = f1_score(
            y_true,
            predictions,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )

        precision, recall, f1_values, support = (
            precision_recall_fscore_support(
                y_true,
                predictions,
                labels=[0, 1, 2, 3],
                zero_division=0
            )
        )

        metric_rows.append({
            "seed": int(seed),
            "condition": condition,
            "holdout_records": 2000,
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1)
        })

        for class_id in range(4):
            per_class_rows.append({
                "seed": int(seed),
                "condition": condition,
                "class_id": class_id,
                "class_name": class_names[class_id],
                "precision": float(precision[class_id]),
                "recall": float(recall[class_id]),
                "f1": float(f1_values[class_id]),
                "support": int(support[class_id])
            })

metrics_df = pd.DataFrame(metric_rows)
per_class_df = pd.DataFrame(per_class_rows)

assert len(metrics_df) == 6
assert len(per_class_df) == 24

# Paired improvements in percentage points / raw F1 units.
paired_rows = []

for seed in SEEDS:
    baseline = metrics_df.loc[
        (metrics_df["seed"] == seed)
        & (metrics_df["condition"] == "baseline")
    ].iloc[0]

    corrected = metrics_df.loc[
        (metrics_df["seed"] == seed)
        & (metrics_df["condition"] == "corrected")
    ].iloc[0]

    paired_rows.append({
        "seed": seed,
        "baseline_macro_f1": float(baseline["macro_f1"]),
        "corrected_macro_f1": float(corrected["macro_f1"]),
        "macro_f1_difference": float(
            corrected["macro_f1"] - baseline["macro_f1"]
        ),
        "baseline_accuracy": float(baseline["accuracy"]),
        "corrected_accuracy": float(corrected["accuracy"]),
        "accuracy_difference_pp": float(
            100 * (
                corrected["accuracy"]
                - baseline["accuracy"]
            )
        )
    })

paired_df = pd.DataFrame(paired_rows)

# Save aggregate results in the private evaluation directory.
def save_immutable_csv(path, frame):
    content = frame.to_csv(
        index=False,
        lineterminator="\n"
    )

    if path.exists():
        assert path.read_text(encoding="utf-8") == content
        return "EXISTING_VERIFIED"

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content, encoding="utf-8")
    os.replace(tmp, path)

    return "NEW_SAVED"

metrics_status = save_immutable_csv(
    EVAL_DIR / "holdout_model_metrics_v1.csv",
    metrics_df
)

per_class_status = save_immutable_csv(
    EVAL_DIR / "holdout_per_class_metrics_v1.csv",
    per_class_df
)

paired_status = save_immutable_csv(
    EVAL_DIR / "holdout_paired_effects_v1.csv",
    paired_df
)

print("\nMODEL-LEVEL HOLDOUT RESULTS")
print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nPAIRED HOLDOUT COMPARISON")
print(
    paired_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nSAVED AGGREGATE REPORTS")
print("Model metrics:", metrics_status)
print("Per-class metrics:", per_class_status)
print("Paired effects:", paired_status)

print("\nRESEARCH INTEGRITY")
print("All six trained models evaluated: YES")
print("Frozen original holdout: VERIFIED")
print("Holdout reference labels used for metrics: YES")
print("Holdout used for training or tuning: NO")
print("Original trained models modified: NO")
print("Per-record reference labels exported: NO")
print("Evaluation artifacts saved privately: YES")

print("\nSTATUS: SIX-MODEL FROZEN HOLDOUT EVALUATION PASSED")


STAGE 13 — FROZEN HOLDOUT EVALUATION
Training runs verified: 6/6
Holdout records: 2000
Primary metric: Macro-F1
GPU: Tesla T4
Model training: DISABLED
Hyperparameter tuning: DISABLED



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

baseline_seed_42: PREDICTIONS SAVED


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

corrected_seed_42: PREDICTIONS SAVED


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

baseline_seed_2026: PREDICTIONS SAVED


Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

corrected_seed_2026: PREDICTIONS SAVED

ALL SIX PREDICTION SETS READY
Calculating frozen evaluation metrics...

MODEL-LEVEL HOLDOUT RESULTS
 seed condition  holdout_records  accuracy  macro_f1
   42  baseline             2000  0.812500  0.806759
   42 corrected             2000  0.855000  0.853293
  123  baseline             2000  0.830000  0.826291
  123 corrected             2000  0.858500  0.857178
 2026  baseline             2000  0.828500  0.824367
 2026 corrected             2000  0.860500  0.859104

PAIRED HOLDOUT COMPARISON
 seed  baseline_macro_f1  corrected_macro_f1  macro_f1_difference  baseline_accuracy  corrected_accuracy  accuracy_difference_pp
   42           0.806759            0.853293             0.046533           0.812500            0.855000                4.250000
  123           0.826291            0.857178             0.030887           0.830000            0.858500                2.850000
 2026           0.824367            0.859104             0.034737          

In [4]:

# STAGE 13 — STEP 16
# Frozen holdout: paired statistical and per-class analysis.
# No training, inference, threshold tuning, or model selection.

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"
EVAL_DIR = ROOT / "holdout_evaluation_v1"

assert "EVAL_HOLDOUT" in globals(), (
    "Restore the frozen holdout using Step 15A first."
)

with (
    ROOT / "downstream_training_protocol_v1.json"
).open(encoding="utf-8") as f:
    protocol = json.load(f)

with (
    BASE / "configs/holdout_partition_manifest_v1.json"
).open(encoding="utf-8") as f:
    partition_manifest = json.load(f)

with (
    EVAL_DIR / "holdout_model_metrics_v1.csv"
).open(encoding="utf-8") as f:
    saved_metrics = pd.read_csv(f)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

def id_fingerprint(ids):
    payload = "\n".join(
        str(int(v)) for v in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

holdout = (
    EVAL_HOLDOUT
    .sort_values("record_id")
    .reset_index(drop=True)
)

ids = holdout["record_id"].to_numpy(dtype=np.int64)
y = holdout["label"].to_numpy(dtype=np.int64)

assert len(ids) == 2000
assert np.isin(y, [0, 1, 2, 3]).all()

holdout_spec = partition_manifest["partitions"]["holdout"]

assert id_fingerprint(ids) == holdout_spec["id_sha256"]
assert protocol["seeds"] == [42, 123, 2026]
assert protocol["evaluation_plan"]["primary_metric"] == (
    "holdout_macro_f1"
)

SEEDS = [42, 123, 2026]
CLASSES = protocol["class_names"]
PROTOCOL_HASH = sha256_file(
    ROOT / "downstream_training_protocol_v1.json"
)

predictions = {}

# Load immutable saved predictions and verify provenance.
for seed in SEEDS:
    for condition in ("baseline", "corrected"):
        key = (seed, condition)
        stem = f"{condition}_seed_{seed}"

        path = EVAL_DIR / f"predictions_{stem}.csv"
        manifest_path = (
            EVAL_DIR / f"predictions_{stem}_manifest.json"
        )

        assert path.is_file()
        assert manifest_path.is_file()

        with manifest_path.open(encoding="utf-8") as f:
            manifest = json.load(f)

        assert manifest["seed"] == seed
        assert manifest["condition"] == condition
        assert manifest["protocol_sha256"] == PROTOCOL_HASH
        assert manifest["holdout_id_sha256"] == (
            holdout_spec["id_sha256"]
        )
        assert manifest["prediction_file_sha256"] == (
            sha256_file(path)
        )
        assert manifest["reference_labels_saved"] is False

        frame = pd.read_csv(path)

        assert list(frame.columns) == [
            "record_id", "prediction"
        ]
        assert len(frame) == len(ids)
        assert np.array_equal(
            frame["record_id"].to_numpy(dtype=np.int64),
            ids
        )

        pred = frame["prediction"].to_numpy(dtype=np.int64)
        assert np.isin(pred, [0, 1, 2, 3]).all()

        predictions[key] = pred

        # Independently reproduce the previously saved metrics.
        previous = saved_metrics.loc[
            (saved_metrics["seed"] == seed)
            & (saved_metrics["condition"] == condition)
        ]

        assert len(previous) == 1

        assert np.isclose(
            float(previous.iloc[0]["macro_f1"]),
            f1_score(
                y, pred,
                labels=[0, 1, 2, 3],
                average="macro",
                zero_division=0
            ),
            atol=1e-12
        )

        assert np.isclose(
            float(previous.iloc[0]["accuracy"]),
            accuracy_score(y, pred),
            atol=1e-12
        )

# Calculate paired seed-level differences.
paired_rows = []
class_rows = []

for seed in SEEDS:
    base = predictions[(seed, "baseline")]
    corr = predictions[(seed, "corrected")]

    base_correct = base == y
    corr_correct = corr == y

    helped = int((~base_correct & corr_correct).sum())
    harmed = int((base_correct & ~corr_correct).sum())

    discordant = helped + harmed

    p_value = (
        float(
            binomtest(
                helped,
                n=discordant,
                p=0.5,
                alternative="two-sided"
            ).pvalue
        )
        if discordant else 1.0
    )

    base_macro = f1_score(
        y, base,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )

    corr_macro = f1_score(
        y, corr,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )

    paired_rows.append({
        "seed": seed,
        "baseline_macro_f1": float(base_macro),
        "corrected_macro_f1": float(corr_macro),
        "macro_f1_gain": float(corr_macro - base_macro),
        "accuracy_gain_pp": float(
            100 * (
                accuracy_score(y, corr)
                - accuracy_score(y, base)
            )
        ),
        "helped_records": helped,
        "harmed_records": harmed,
        "discordant_records": discordant,
        "mcnemar_exact_p": p_value
    })

    _, _, base_f1, supports = (
        precision_recall_fscore_support(
            y, base,
            labels=[0, 1, 2, 3],
            zero_division=0
        )
    )

    _, _, corr_f1, _ = (
        precision_recall_fscore_support(
            y, corr,
            labels=[0, 1, 2, 3],
            zero_division=0
        )
    )

    for class_id, class_name in enumerate(CLASSES):
        class_rows.append({
            "seed": seed,
            "class_id": class_id,
            "class_name": class_name,
            "support": int(supports[class_id]),
            "baseline_f1": float(base_f1[class_id]),
            "corrected_f1": float(corr_f1[class_id]),
            "f1_gain": float(
                corr_f1[class_id] - base_f1[class_id]
            )
        })

paired = pd.DataFrame(paired_rows)
per_class = pd.DataFrame(class_rows)

# Holm-adjust the three seed-level McNemar p-values.
pvalues = paired["mcnemar_exact_p"].to_numpy()
order = np.argsort(pvalues)
adjusted = np.empty(3, dtype=float)
running = 0.0

for rank, index in enumerate(order):
    running = max(
        running,
        min(1.0, (3 - rank) * float(pvalues[index]))
    )
    adjusted[index] = running

paired["mcnemar_holm_p"] = adjusted

# Seed-to-seed variability (descriptive; only three seeds).
mean_gain = float(paired["macro_f1_gain"].mean())
sd_gain = float(paired["macro_f1_gain"].std(ddof=1))

mean_accuracy_gain = float(
    paired["accuracy_gain_pp"].mean()
)
sd_accuracy_gain = float(
    paired["accuracy_gain_pp"].std(ddof=1)
)

# Paired holdout bootstrap, with the same resampled records
# used for all three matched seed comparisons.
#
# This interval is conditional on these three trained seeds.
# It is NOT an independent uncertainty interval over new seeds.
def macro_f1_fast(truth, pred):
    matrix = np.bincount(
        4 * truth + pred,
        minlength=16
    ).reshape(4, 4)

    tp = np.diag(matrix).astype(float)
    denominator = (
        matrix.sum(axis=0)
        + matrix.sum(axis=1)
    ).astype(float)

    scores = np.divide(
        2 * tp,
        denominator,
        out=np.zeros(4, dtype=float),
        where=denominator > 0
    )

    return float(scores.mean())

BOOTSTRAPS = 5000
BOOTSTRAP_SEED = 20261111
rng = np.random.default_rng(BOOTSTRAP_SEED)

bootstrap_gains = np.empty(
    BOOTSTRAPS, dtype=float
)

for iteration in range(BOOTSTRAPS):
    sampled_indices = rng.integers(
        0, len(y), size=len(y)
    )

    sampled_truth = y[sampled_indices]
    differences = []

    for seed in SEEDS:
        base_sampled = predictions[
            (seed, "baseline")
        ][sampled_indices]

        corr_sampled = predictions[
            (seed, "corrected")
        ][sampled_indices]

        differences.append(
            macro_f1_fast(sampled_truth, corr_sampled)
            - macro_f1_fast(sampled_truth, base_sampled)
        )

    bootstrap_gains[iteration] = np.mean(differences)

ci_low, ci_high = np.percentile(
    bootstrap_gains,
    [2.5, 97.5]
)

# Summarize class effects without exposing record-level labels.
class_summary = (
    per_class.groupby(
        ["class_id", "class_name"],
        as_index=False
    )
    .agg(
        support=("support", "first"),
        baseline_f1_mean=("baseline_f1", "mean"),
        corrected_f1_mean=("corrected_f1", "mean"),
        f1_gain_mean=("f1_gain", "mean"),
        f1_gain_sd=("f1_gain", "std")
    )
    .sort_values("class_id")
)

assert len(paired) == 3
assert len(per_class) == 12
assert len(class_summary) == 4

# Save aggregated statistics only. No texts or true labels.
def save_immutable(path, content):
    if path.exists():
        assert path.read_text(
            encoding="utf-8"
        ) == content, (
            f"Existing report differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        content,
        encoding="utf-8"
    )
    os.replace(temporary, path)
    return "NEW_SAVED"

save_immutable(
    EVAL_DIR / "holdout_seed_statistical_tests_v1.csv",
    paired.to_csv(index=False, lineterminator="\n")
)

save_immutable(
    EVAL_DIR / "holdout_per_class_comparison_v1.csv",
    per_class.to_csv(index=False, lineterminator="\n")
)

save_immutable(
    EVAL_DIR / "holdout_class_summary_v1.csv",
    class_summary.to_csv(index=False, lineterminator="\n")
)

report = {
    "analysis_version": 1,
    "status": "PAIRED_HOLDOUT_ANALYSIS_COMPLETED",
    "seeds": SEEDS,
    "holdout_records": 2000,
    "primary_metric": "macro_f1",
    "mean_macro_f1_gain": mean_gain,
    "sd_macro_f1_gain_across_seeds": sd_gain,
    "mean_accuracy_gain_percentage_points": (
        mean_accuracy_gain
    ),
    "sd_accuracy_gain_pp_across_seeds": (
        sd_accuracy_gain
    ),
    "macro_f1_gain_paired_bootstrap_95_ci": [
        float(ci_low), float(ci_high)
    ],
    "bootstrap_resamples": BOOTSTRAPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "bootstrap_unit": "HOLDOUT_RECORD",
    "bootstrap_seed_handling": (
        "FIXED_THREE_SEEDS_WITH_SHARED_RESAMPLED_RECORDS"
    ),
    "statistical_test": (
        "EXACT_MCNEMAR_PER_SEED_WITH_HOLM_ADJUSTMENT"
    ),
    "analysis_interpretation": (
        "Conditional on these three trained models per "
        "condition and this held-out data distribution; "
        "does not account for new model architectures, "
        "new corpora, or independent training-seed "
        "uncertainty in the bootstrap interval."
    ),
    "model_training_performed": False,
    "hyperparameter_tuning_performed": False,
    "reference_labels_exported": False
}

save_immutable(
    EVAL_DIR / "holdout_statistical_report_v1.json",
    json.dumps(report, indent=2) + "\n"
)

print("STAGE 13 — STEP 16: PAIRED STATISTICAL ANALYSIS")

print("\nSEED-LEVEL PAIRED RESULTS")
print(
    paired.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)

print("\nPER-CLASS AVERAGE F1")
print(
    class_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nAGGREGATE EFFECT")
print(f"Mean Macro-F1 gain: {mean_gain:+.6f}")
print(f"Seed SD of gain: {sd_gain:.6f}")
print(
    f"Paired bootstrap 95% CI: "
    f"[{ci_low:+.6f}, {ci_high:+.6f}]"
)
print(
    f"Mean accuracy gain: "
    f"{mean_accuracy_gain:+.4f} percentage points"
)
print(
    f"Seed SD of accuracy gain: "
    f"{sd_accuracy_gain:.4f} percentage points"
)

print("\nLIMITATIONS")
print("Only 3 matched random seeds: YES")
print("Bootstrap conditions on these seeds: YES")
print("Multiple McNemar tests Holm-adjusted: YES")
print("Other datasets evaluated: NO")
print("Hyperparameters changed after holdout: NO")
print("Reference labels exported: NO")

print("\nSTATUS: PAIRED STATISTICAL ANALYSIS PASSED")


STAGE 13 — STEP 16: PAIRED STATISTICAL ANALYSIS

SEED-LEVEL PAIRED RESULTS
 seed  baseline_macro_f1  corrected_macro_f1  macro_f1_gain  accuracy_gain_pp  helped_records  harmed_records  discordant_records  mcnemar_exact_p  mcnemar_holm_p
   42           0.806759            0.853293      0.0465334              4.25             102              17                 119      5.86012e-16     1.75804e-15
  123           0.826291            0.857178      0.0308873              2.85              72              15                  87      4.26054e-10     4.26054e-10
 2026           0.824367            0.859104      0.0347373               3.2              80              16                  96      2.07441e-11     4.14882e-11

PER-CLASS AVERAGE F1
 class_id class_name  support  baseline_f1_mean  corrected_f1_mean  f1_gain_mean  f1_gain_sd
        0      World      476          0.824873           0.837224      0.012351    0.005129
        1     Sports      503          0.956581           0.96547

In [5]:

# STAGE 13 — STEP 17
# Verify and freeze aggregate downstream research results.
# No training, inference, tuning, or reference-label access.

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/S_SLTE_Thesis"
) / "outputs/s_slte_downstream_v1"

EVAL_DIR = ROOT / "holdout_evaluation_v1"

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

files = {
    "protocol": ROOT / "downstream_training_protocol_v1.json",
    "training_audit": ROOT / "final_training_audit_v1.json",
    "metrics": EVAL_DIR / "holdout_model_metrics_v1.csv",
    "paired": EVAL_DIR / "holdout_seed_statistical_tests_v1.csv",
    "classes": EVAL_DIR / "holdout_class_summary_v1.csv",
    "statistics": EVAL_DIR / "holdout_statistical_report_v1.json"
}

assert all(path.is_file() for path in files.values())

with files["protocol"].open(encoding="utf-8") as f:
    protocol = json.load(f)

with files["training_audit"].open(encoding="utf-8") as f:
    audit = json.load(f)

with files["statistics"].open(encoding="utf-8") as f:
    statistics = json.load(f)

metrics = pd.read_csv(files["metrics"])
paired = pd.read_csv(files["paired"])
classes = pd.read_csv(files["classes"])

assert audit["status"] == "ALL_SIX_TRAINING_RUNS_AUDITED"
assert audit["completed_runs"] == 6
assert audit["protocol_sha256"] == sha256_file(
    files["protocol"]
)

assert statistics["status"] == (
    "PAIRED_HOLDOUT_ANALYSIS_COMPLETED"
)

assert protocol["evaluation_plan"]["primary_metric"] == (
    "holdout_macro_f1"
)

assert protocol["evaluation_plan"]["holdout_used_for_tuning"] is False
assert sorted(metrics["seed"].unique().tolist()) == [42, 123, 2026]
assert len(metrics) == 6
assert len(paired) == 3
assert len(classes) == 4

assert set(metrics["condition"]) == {
    "baseline", "corrected"
}
assert (metrics["holdout_records"] == 2000).all()
assert (classes["support"].sum() == 2000)

baseline = (
    metrics[metrics["condition"] == "baseline"]
    .sort_values("seed")
    .reset_index(drop=True)
)

corrected = (
    metrics[metrics["condition"] == "corrected"]
    .sort_values("seed")
    .reset_index(drop=True)
)

paired = paired.sort_values("seed").reset_index(drop=True)

assert np.array_equal(
    baseline["seed"].to_numpy(),
    corrected["seed"].to_numpy()
)
assert np.array_equal(
    paired["seed"].to_numpy(),
    baseline["seed"].to_numpy()
)

f1_gains = (
    corrected["macro_f1"].to_numpy()
    - baseline["macro_f1"].to_numpy()
)

accuracy_gains_pp = 100 * (
    corrected["accuracy"].to_numpy()
    - baseline["accuracy"].to_numpy()
)

assert np.allclose(
    f1_gains,
    paired["macro_f1_gain"].to_numpy(),
    atol=1e-12
)

assert np.allclose(
    accuracy_gains_pp,
    paired["accuracy_gain_pp"].to_numpy(),
    atol=1e-12
)

mean_gain = float(np.mean(f1_gains))
mean_accuracy_gain = float(np.mean(accuracy_gains_pp))

assert np.isclose(
    mean_gain,
    statistics["mean_macro_f1_gain"],
    atol=1e-12
)

assert np.isclose(
    mean_accuracy_gain,
    statistics["mean_accuracy_gain_percentage_points"],
    atol=1e-12
)

assert np.all(f1_gains > 0)
assert np.all(accuracy_gains_pp > 0)

ci_low, ci_high = statistics[
    "macro_f1_gain_paired_bootstrap_95_ci"
]

assert ci_low > 0
assert ci_low < mean_gain < ci_high

assert (paired["mcnemar_holm_p"] < 0.001).all()

class_mean_gains = classes["f1_gain_mean"].to_numpy()
assert np.isclose(
    class_mean_gains.mean(),
    mean_gain,
    atol=1e-10
)

baseline_mean = float(baseline["macro_f1"].mean())
corrected_mean = float(corrected["macro_f1"].mean())

baseline_sd = float(baseline["macro_f1"].std(ddof=1))
corrected_sd = float(corrected["macro_f1"].std(ddof=1))

# Human-readable summary with only aggregate statistics.
lines = [
    "# S-SLTE — Frozen Downstream Holdout Results",
    "",
    "**Experiment:** AG News text classification with DistilBERT.",
    "**Method:** Frozen semantic-neighborhood label correction (S-SLTE v1).",
    "**Evaluation:** 2,000 frozen internal holdout records.",
    "**Random seeds:** 42, 123, 2026.",
    "**Training records per run:** 13,621.",
    "**Training label changes:** 705 of 13,621.",
    "**Trained models:** 6; three matched seed pairs.",
    "",
    "## Primary Metric",
    "",
    "| Condition | Mean Macro-F1 | SD across seeds |",
    "|---|---:|---:|",
    f"| Baseline | {baseline_mean:.6f} | {baseline_sd:.6f} |",
    f"| S-SLTE corrected | {corrected_mean:.6f} | {corrected_sd:.6f} |",
    "",
    f"**Mean paired Macro-F1 gain:** +{mean_gain:.6f}",
    f"**SD of paired gain across seeds:** {np.std(f1_gains, ddof=1):.6f}",
    f"**Paired holdout bootstrap 95% CI:** "
    f"[+{ci_low:.6f}, +{ci_high:.6f}]",
    f"**Mean accuracy improvement:** "
    f"+{mean_accuracy_gain:.4f} percentage points.",
    "",
    "## Per-class Mean F1",
    "",
    "| Class | Baseline | Corrected | Gain |",
    "|---|---:|---:|---:|"
]

for row in classes.sort_values("class_id").itertuples(
    index=False
):
    lines.append(
        f"| {row.class_name} | "
        f"{row.baseline_f1_mean:.6f} | "
        f"{row.corrected_f1_mean:.6f} | "
        f"+{row.f1_gain_mean:.6f} |"
    )

lines.extend([
    "",
    "## Statistical Interpretation",
    "",
    "- All three matched seed pairs favor corrected labels.",
    "- Exact McNemar tests were performed separately on paired "
    "accuracy outcomes; all three Holm-adjusted p-values are below 0.001.",
    "- The paired bootstrap interval concerns the average Macro-F1 "
    "gain conditional on these three trained seed pairs.",
    "- McNemar tests do not directly test Macro-F1 differences.",
    "",
    "## Limitations",
    "",
    "- One dataset, one model architecture, and three training seeds.",
    "- The internal holdout is not evidence of external-domain generalization.",
    "- The Stage 12 development evaluation was not fully independent "
    "of earlier full-development neighborhood exploration.",
    "- The 705 automatic training-label replacements were not "
    "individually verified against trusted labels.",
    "- Strong Sci/Tech-directed correction imbalance requires caution.",
    "- No further tuning or model selection should use this holdout.",
    "",
    "## Research Integrity",
    "",
    "- All six training runs completed before holdout evaluation.",
    "- Training targets used original LLM labels or frozen S-SLTE labels.",
    "- Holdout labels were not used for training or hyperparameter tuning.",
    "- This document contains aggregate statistics only.",
    "- No record-level reference labels or article texts are exported.",
    ""
])

report_text = "\n".join(lines)
report_path = EVAL_DIR / "holdout_results_summary_v1.md"

if report_path.exists():
    assert report_path.read_text(
        encoding="utf-8"
    ) == report_text
    report_status = "EXISTING_VERIFIED"
else:
    temp_path = report_path.with_suffix(".md.tmp")
    temp_path.write_text(report_text, encoding="utf-8")
    os.replace(temp_path, report_path)
    report_status = "NEW_SAVED"

print("STAGE 13 — FINAL RESULTS VERIFICATION")
print()
print("Training runs:", audit["completed_runs"])
print("Evaluation records:", 2000)
print("Baseline mean Macro-F1:", f"{baseline_mean:.6f}")
print("Corrected mean Macro-F1:", f"{corrected_mean:.6f}")
print("Mean Macro-F1 gain:", f"+{mean_gain:.6f}")
print(
    "Mean accuracy gain:",
    f"+{mean_accuracy_gain:.4f} pp"
)
print(
    "Paired bootstrap CI:",
    f"[+{ci_low:.6f}, +{ci_high:.6f}]"
)
print("All three paired effects positive: YES")
print("Statistical report consistent: YES")
print("Per-class results consistent: YES")

print("\nREPORT")
print("Status:", report_status)
print("Path:", report_path)

print("\nRESEARCH INTEGRITY")
print("Training performed: NO")
print("New holdout inference performed: NO")
print("Reference labels accessed: NO")
print("Frozen protocol changed: NO")

print("\nSTATUS: STAGE 13 FINAL RESULTS VERIFIED")


STAGE 13 — FINAL RESULTS VERIFICATION

Training runs: 6
Evaluation records: 2000
Baseline mean Macro-F1: 0.819139
Corrected mean Macro-F1: 0.856525
Mean Macro-F1 gain: +0.037386
Mean accuracy gain: +3.4333 pp
Paired bootstrap CI: [+0.028502, +0.046546]
All three paired effects positive: YES
Statistical report consistent: YES
Per-class results consistent: YES

REPORT
Status: NEW_SAVED
Path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/holdout_evaluation_v1/holdout_results_summary_v1.md

RESEARCH INTEGRITY
Training performed: NO
New holdout inference performed: NO
Reference labels accessed: NO
Frozen protocol changed: NO

STATUS: STAGE 13 FINAL RESULTS VERIFIED
